# DPF Rebuttal Experiments

# 30-run paired evaluation for Table 5:

In [6]:
"""

Alternative interpretation functionals under the same Gibbs mapping family
on MVSA-Single.

Protocol
--------
1. Load fixed upstream predictive probabilities.
2. Select beta ONCE for each interpretation using the validation set.
3. Freeze the selected beta values.
4. Generate the SAME stratified 2x bootstrap training sample for all
   interpretation functionals within each run.
5. Retrain the downstream XGBoost classifier for every method/run.
6. Evaluate all methods on the SAME untouched test set.
7. Report:
      - Mean weighted F1
      - Standard deviation
      - Mean paired difference
      - 95% bootstrap CI of paired difference
      - Paired t-test
      - Wilcoxon signed-rank test

This isolates interpretation-functional differences while controlling
the Gibbs mapping family and paired resampling variation.
"""

import pickle
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import f1_score
from scipy.stats import ttest_rel, wilcoxon


# ============================================================
# Configuration
# ============================================================

PKL_PATH = "../cache/dpf_mvsa_single_results.pkl"

TOPK = 2
DELTA = 1e-2

N_RUNS = 30
BOOTSTRAP_FACTOR = 2

BASE_SEED = 42

BETA_CANDIDATES = [
    0.0, 0.1, 0.25, 0.5, 0.8, 1.0,
    1.1, 1.2, 1.5, 2.0, 3.0, 5.0
]

METHODS = [
    ("Uniform", "uniform"),
    ("Shannon", "shannon"),
    ("Renyi-2", "renyi2"),
    ("Practical ITP", "itp"),
]


# ============================================================
# Load probabilities
# ============================================================

print("=" * 80)
print("Loading:", PKL_PATH)
print("=" * 80)

with open(PKL_PATH, "rb") as f:
    results = pickle.load(f)

train_probs_text = np.asarray(results["train_probs_text"])
train_probs_image = np.asarray(results["train_probs_image"])

val_probs_text = np.asarray(results["val_probs_text"])
val_probs_image = np.asarray(results["val_probs_image"])

test_probs_text = np.asarray(results["test_probs_text"])
test_probs_image = np.asarray(results["test_probs_image"])

y_train = np.asarray(results["y_train_int"])
y_val = np.asarray(results["y_val_int"])
y_test = np.asarray(results["y_test_int"])

print(f"Train: {train_probs_text.shape}")
print(f"Val:   {val_probs_text.shape}")
print(f"Test:  {test_probs_text.shape}")


# ============================================================
# Potential functionals
# ============================================================

def compute_potential(probs, method, topk=2, delta=1e-2):

    probs = np.asarray(probs)

    if method == "uniform":
        return np.zeros(len(probs))

    if method == "shannon":
        return -np.sum(
            probs * np.log(probs + 1e-12),
            axis=1
        )

    if method == "renyi2":
        return -np.log(
            np.sum(probs ** 2, axis=1) + 1e-12
        )

    if method == "itp":
        k = min(topk, probs.shape[1])

        topk_probs = np.sort(probs, axis=1)[:, -k:]

        sum_topk_sq = np.sum(
            topk_probs ** 2,
            axis=1
        )

        max_topk_sq = np.max(
            topk_probs ** 2,
            axis=1
        )

        return -np.log(
            sum_topk_sq
            + delta * max_topk_sq
            + 1e-12
        )

    raise ValueError(f"Unknown method: {method}")


# ============================================================
# Gibbs realization
# ============================================================

def gibbs_fusion(
    probs_text,
    probs_image,
    method,
    beta,
    topk=2,
    delta=1e-2
):

    n = len(probs_text)

    if method == "uniform" or beta == 0:

        w_text = np.full((n, 1), 0.5)
        w_image = np.full((n, 1), 0.5)

    else:

        e_text = compute_potential(
            probs_text,
            method,
            topk,
            delta
        )

        e_image = compute_potential(
            probs_image,
            method,
            topk,
            delta
        )

        energies = np.column_stack([
            e_text,
            e_image
        ])

        # Numerically stable Gibbs softmin
        logits = -beta * energies
        logits -= np.max(
            logits,
            axis=1,
            keepdims=True
        )

        weights = np.exp(logits)
        weights /= np.sum(
            weights,
            axis=1,
            keepdims=True
        )

        w_text = weights[:, 0:1]
        w_image = weights[:, 1:2]

    return np.concatenate(
        [
            w_text * probs_text,
            w_image * probs_image
        ],
        axis=1
    )


# ============================================================
# XGBoost
# ============================================================

def make_classifier(seed):

    return xgb.XGBClassifier(
        n_estimators=350,
        max_depth=5,
        random_state=seed,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1
    )


# ============================================================
# STEP 1
# Select beta ONCE using validation set
# ============================================================

def select_beta(method):

    if method == "uniform":
        return 0.0, None

    best_beta = None
    best_f1 = -np.inf

    for beta in BETA_CANDIDATES:

        X_train = gibbs_fusion(
            train_probs_text,
            train_probs_image,
            method,
            beta,
            TOPK,
            DELTA
        )

        X_val = gibbs_fusion(
            val_probs_text,
            val_probs_image,
            method,
            beta,
            TOPK,
            DELTA
        )

        clf = make_classifier(BASE_SEED)
        clf.fit(X_train, y_train)

        pred = clf.predict(X_val)

        score = f1_score(
            y_val,
            pred,
            average="weighted"
        )

        print(
            f"    beta={beta:<5} "
            f"Val W-F1={score:.4f}"
        )

        if score > best_f1:
            best_f1 = score
            best_beta = beta

    return best_beta, best_f1


print("\n" + "=" * 80)
print("STEP 1: VALIDATION BETA SELECTION")
print("=" * 80)

selected_beta = {}

for display_name, method in METHODS:

    print(f"\n{display_name}")

    beta_star, val_score = select_beta(method)

    selected_beta[method] = beta_star

    if val_score is None:
        print("  beta* = 0.0 (uniform)")
    else:
        print(
            f"  SELECTED beta*={beta_star} "
            f"| Val W-F1={val_score:.4f}"
        )


print("\nFrozen beta values:")
for _, method in METHODS:
    print(
        f"  {method:<10}: "
        f"{selected_beta[method]}"
    )


# ============================================================
# Stratified 2x bootstrap
# ============================================================

def stratified_bootstrap_indices(
    labels,
    factor,
    rng
):
    """
    Sample WITH replacement within each class.

    For factor=2, each class contributes 2x its
    original number of training observations.

    The returned index array is shared by ALL
    interpretation methods for a given run.
    """

    sampled = []

    for cls in np.unique(labels):

        cls_idx = np.flatnonzero(labels == cls)

        n_sample = int(
            len(cls_idx) * factor
        )

        boot_idx = rng.choice(
            cls_idx,
            size=n_sample,
            replace=True
        )

        sampled.append(boot_idx)

    sampled = np.concatenate(sampled)

    # Shuffle while preserving sampled composition
    rng.shuffle(sampled)

    return sampled


# ============================================================
# Precompute fused representations
#
# beta is frozen, so these do not change across runs.
# Runs differ ONLY through paired bootstrap training samples
# and downstream classifier random seed.
# ============================================================

print("\n" + "=" * 80)
print("PRECOMPUTING FUSED REPRESENTATIONS")
print("=" * 80)

fused_train = {}
fused_test = {}

for display_name, method in METHODS:

    beta = selected_beta[method]

    fused_train[method] = gibbs_fusion(
        train_probs_text,
        train_probs_image,
        method,
        beta,
        TOPK,
        DELTA
    )

    fused_test[method] = gibbs_fusion(
        test_probs_text,
        test_probs_image,
        method,
        beta,
        TOPK,
        DELTA
    )

    print(
        f"{display_name:<15} "
        f"beta={beta}"
    )


# ============================================================
# STEP 2
# 30 paired bootstrap runs
# ============================================================

print("\n" + "=" * 80)
print(
    f"STEP 2: {N_RUNS} PAIRED RUNS "
    f"({BOOTSTRAP_FACTOR}x STRATIFIED BOOTSTRAP)"
)
print("=" * 80)

run_rows = []

for run in range(N_RUNS):

    seed = BASE_SEED + run

    rng = np.random.default_rng(seed)

    # IMPORTANT:
    # Same indices used for every functional
    boot_idx = stratified_bootstrap_indices(
        y_train,
        BOOTSTRAP_FACTOR,
        rng
    )

    y_boot = y_train[boot_idx]

    run_result = {
        "Run": run + 1,
        "Seed": seed
    }

    print(
        f"\nRun {run + 1:02d}/{N_RUNS} "
        f"| seed={seed} "
        f"| bootstrap N={len(boot_idx)}"
    )

    for display_name, method in METHODS:

        X_boot = fused_train[method][boot_idx]
        X_test = fused_test[method]

        clf = make_classifier(seed)

        clf.fit(
            X_boot,
            y_boot
        )

        pred = clf.predict(X_test)

        wf1 = f1_score(
            y_test,
            pred,
            average="weighted"
        )

        run_result[method] = wf1

        print(
            f"  {display_name:<15} "
            f"W-F1={wf1:.4f}"
        )

    run_rows.append(run_result)


df_runs = pd.DataFrame(run_rows)


# ============================================================
# STEP 3
# Summary statistics
# ============================================================

print("\n" + "=" * 80)
print("30-RUN SUMMARY")
print("=" * 80)

summary_rows = []

for display_name, method in METHODS:

    values = df_runs[method].to_numpy()

    summary_rows.append({
        "Functional": display_name,
        "Method": method,
        "beta*": selected_beta[method],
        "Mean_WF1": np.mean(values),
        "SD_WF1": np.std(values, ddof=1),
        "Min_WF1": np.min(values),
        "Max_WF1": np.max(values)
    })

df_summary = pd.DataFrame(summary_rows)

print(
    df_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# Paired comparison utility
# ============================================================

def paired_comparison(
    method_a,
    method_b,
    n_ci_boot=10000,
    seed=2027
):
    """
    Difference = method_a - method_b
    """

    a = df_runs[method_a].to_numpy()
    b = df_runs[method_b].to_numpy()

    diff = a - b

    mean_diff = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)

    # Paired t-test
    t_stat, t_p = ttest_rel(a, b)

    # Wilcoxon
    try:
        w_stat, w_p = wilcoxon(
            a,
            b,
            alternative="two-sided"
        )
    except ValueError:
        w_stat = np.nan
        w_p = np.nan

    # Bootstrap CI over the 30 paired differences
    rng = np.random.default_rng(seed)

    boot_means = np.empty(n_ci_boot)

    n = len(diff)

    for i in range(n_ci_boot):

        sampled_diff = rng.choice(
            diff,
            size=n,
            replace=True
        )

        boot_means[i] = np.mean(
            sampled_diff
        )

    ci_low, ci_high = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    return {
        "Comparison":
            f"{method_a} - {method_b}",

        "Mean_Difference":
            mean_diff,

        "SD_Difference":
            sd_diff,

        "CI95_Low":
            ci_low,

        "CI95_High":
            ci_high,

        "Paired_t":
            t_stat,

        "Paired_t_p":
            t_p,

        "Wilcoxon_W":
            w_stat,

        "Wilcoxon_p":
            w_p
    }


# ============================================================
# STEP 4
# Pairwise tests
# ============================================================

print("\n" + "=" * 80)
print("PAIRED STATISTICAL COMPARISONS")
print("=" * 80)

comparisons = [
    ("itp", "uniform"),
    ("itp", "shannon"),
    ("itp", "renyi2"),
    ("shannon", "uniform"),
    ("renyi2", "uniform"),
]

comparison_rows = []

for a, b in comparisons:

    result = paired_comparison(a, b)

    comparison_rows.append(result)

df_comparisons = pd.DataFrame(
    comparison_rows
)

print(
    df_comparisons.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# Compact manuscript/rebuttal output
# ============================================================

print("\n" + "=" * 80)
print("COMPACT TABLE 5 OUTPUT")
print("=" * 80)

uniform_mean = df_runs["uniform"].mean()

for display_name, method in METHODS:

    values = df_runs[method].to_numpy()

    mean = values.mean()
    sd = values.std(ddof=1)

    delta = mean - uniform_mean

    print(
        f"{display_name:<15} "
        f"beta*={selected_beta[method]:<5} "
        f"W-F1={mean:.4f} ± {sd:.4f} "
        f"| ΔUniform={delta:+.4f}"
    )


print("\nITP paired comparisons:")

for opponent in [
    "uniform",
    "shannon",
    "renyi2"
]:

    r = paired_comparison(
        "itp",
        opponent
    )

    print(
        f"  ITP vs {opponent:<8}: "
        f"Δ={r['Mean_Difference']:+.4f}, "
        f"95% CI "
        f"[{r['CI95_Low']:+.4f}, "
        f"{r['CI95_High']:+.4f}], "
        f"paired-t p={r['Paired_t_p']:.6g}, "
        f"Wilcoxon p={r['Wilcoxon_p']:.6g}"
    )




Loading: ../cache/dpf_mvsa_single_results.pkl
Train: (3895, 3)
Val:   (487, 3)
Test:  (487, 3)

STEP 1: VALIDATION BETA SELECTION

Uniform
  beta* = 0.0 (uniform)

Shannon
    beta=0.0   Val W-F1=0.4753
    beta=0.1   Val W-F1=0.4695
    beta=0.25  Val W-F1=0.4565
    beta=0.5   Val W-F1=0.4568
    beta=0.8   Val W-F1=0.4516
    beta=1.0   Val W-F1=0.4481
    beta=1.1   Val W-F1=0.4595
    beta=1.2   Val W-F1=0.4554
    beta=1.5   Val W-F1=0.4698
    beta=2.0   Val W-F1=0.4684
    beta=3.0   Val W-F1=0.4335
    beta=5.0   Val W-F1=0.4625
  SELECTED beta*=0.0 | Val W-F1=0.4753

Renyi-2
    beta=0.0   Val W-F1=0.4753
    beta=0.1   Val W-F1=0.4623
    beta=0.25  Val W-F1=0.4585
    beta=0.5   Val W-F1=0.4730
    beta=0.8   Val W-F1=0.4624
    beta=1.0   Val W-F1=0.4513
    beta=1.1   Val W-F1=0.4478
    beta=1.2   Val W-F1=0.4614
    beta=1.5   Val W-F1=0.4622
    beta=2.0   Val W-F1=0.4511
    beta=3.0   Val W-F1=0.4484
    beta=5.0   Val W-F1=0.4379
  SELECTED beta*=0.0 | Val W-F1=0.47

C:\ProgramData\anaconda3\lib\site-packages\scipy\stats\_morestats.py:3255: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "


       Comparison  Mean_Difference  SD_Difference  CI95_Low  CI95_High  Paired_t  Paired_t_p  Wilcoxon_W  Wilcoxon_p
    itp - uniform         0.017110       0.011308  0.013060   0.020988  8.287509    0.000000    5.000000    0.000000
    itp - shannon         0.017110       0.011308  0.013060   0.020988  8.287509    0.000000    5.000000    0.000000
     itp - renyi2         0.017110       0.011308  0.013060   0.020988  8.287509    0.000000    5.000000    0.000000
shannon - uniform         0.000000       0.000000  0.000000   0.000000       NaN         NaN         NaN         NaN
 renyi2 - uniform         0.000000       0.000000  0.000000   0.000000       NaN         NaN         NaN         NaN

COMPACT TABLE 5 OUTPUT
Uniform         beta*=0.0   W-F1=0.5094 ± 0.0072 | ΔUniform=+0.0000
Shannon         beta*=0.0   W-F1=0.5094 ± 0.0072 | ΔUniform=+0.0000
Renyi-2         beta*=0.0   W-F1=0.5094 ± 0.0072 | ΔUniform=+0.0000
Practical ITP   beta*=1.5   W-F1=0.5265 ± 0.0116 | ΔUniform=+0.0171

I

# Controlled Monte Carlo simulation for Table 5 mechanism analysis

In [7]:
"""
  - Full-support ITP vs Top-k ITP
  - Matched-distribution perturbation: fix Top-k mass, redistribute tail
  - Measure sensitivity of the potential to accumulated tail mass
  - C in {3, 10, 20, 50, 100}, tail-noise levels in {0.0, 0.05, 0.10, 0.20}
  - 10,000 Monte Carlo draws per (C, noise) cell
  - Report mean ± 95% CI of Delta_ITP

This isolates the mechanistic question:
  When decision-relevant dominant mass is held fixed,
  how much does increasing irrelevant tail dimensionality
  alter the potential?

This script does NOT modify any existing code.
"""

import numpy as np
import pandas as pd

# ============================================================
# Configuration
# ============================================================
SEED = 2027
N_DRAWS = 10_000
TOPK = 2
DELTA = 1e-2

C_LIST = [3, 10, 20, 50, 100]
TAIL_NOISE_LEVELS = [0.0, 0.05, 0.10, 0.20]

rng = np.random.default_rng(SEED)


# ============================================================
# Distribution generators
# ============================================================
def generate_matched_distribution(C, topk, tail_noise, rng):
    """
    Generate a base distribution P with:
      - Top-k mass fixed at a concentration level (e.g., 0.90, 0.05)
      - Remaining mass (1 - sum_topk) redistributed over C-k tail classes
      - tail_noise controls the spread of tail mass
    Then perturb ONLY the tail (redistribute uniformly vs. Dirichlet).

    Returns:
        P_clean: original distribution (tail uniform)
        P_tail:  tail-perturbed distribution
    """
    k = min(topk, C)

    # Base Top-k mass: concentrated but not degenerate
    # We generate top-k mass from a Dirichlet with low concentration
    topk_mass = rng.dirichlet(np.ones(k) * 0.3)
    topk_mass = topk_mass * rng.uniform(0.7, 0.95)

    # Remaining mass for tail
    remaining = 1.0 - topk_mass.sum()
    if remaining < 1e-8 or C - k == 0:
        tail_mass = np.array([])
    else:
        # Clean tail: uniform over C-k classes
        tail_mass_clean = np.full(C - k, remaining / (C - k))

        # Tail-perturbed: Dirichlet with concentration controlled by tail_noise
        if tail_noise == 0.0:
            tail_mass_perturbed = tail_mass_clean.copy()
        else:
            alpha = np.ones(C - k) * (1.0 / (tail_noise + 1e-8))
            tail_mass_perturbed = rng.dirichlet(alpha) * remaining

    P_clean = np.concatenate([topk_mass, tail_mass_clean]) if len(tail_mass_clean) > 0 else topk_mass.copy()
    P_tail = np.concatenate([topk_mass, tail_mass_perturbed]) if len(tail_mass_perturbed) > 0 else topk_mass.copy()

    # Ensure non-negative and normalize
    P_clean = np.clip(P_clean, 0, None)
    P_clean /= P_clean.sum() + 1e-12
    P_tail = np.clip(P_tail, 0, None)
    P_tail /= P_tail.sum() + 1e-12

    return P_clean, P_tail


# ============================================================
# Potential functionals
# ============================================================
def itp_full_support(P, delta=1e-2):
    """Full-support ITP: -log(sum_c p_c^2 + delta * max_c p_c^2)"""
    sum_sq = np.sum(P ** 2)
    max_sq = np.max(P ** 2)
    return -np.log(sum_sq + delta * max_sq + 1e-12)


def itp_topk(P, topk=2, delta=1e-2):
    """Top-k ITP: -log(sum_{c in Top-k} p_c^2 + delta * max_{c in Top-k} p_c^2)"""
    k = min(topk, len(P))
    topk_probs = np.sort(P)[-k:]
    sum_sq = np.sum(topk_probs ** 2)
    max_sq = np.max(topk_probs ** 2)
    return -np.log(sum_sq + delta * max_sq + 1e-12)


# ============================================================
# Monte Carlo
# ============================================================
def run_monte_carlo(C, tail_noise, n_draws, topk=2, delta=1e-2):
    """
    For each draw:
      - Generate matched pair (P_clean, P_tail)
      - Compute ITP_full and ITP_topk on both
      - Record |Delta_ITP| = |ITP(clean) - ITP(tail)|
    Return mean and 95% CI of Delta_ITP for each functional.
    """
    delta_full = []
    delta_topk = []

    for _ in range(n_draws):
        P_clean, P_tail = generate_matched_distribution(C, topk, tail_noise, rng)

        # Full-support ITP
        f_clean = itp_full_support(P_clean, delta)
        f_tail = itp_full_support(P_tail, delta)
        delta_full.append(abs(f_clean - f_tail))

        # Top-k ITP
        t_clean = itp_topk(P_clean, topk, delta)
        t_tail = itp_topk(P_tail, topk, delta)
        delta_topk.append(abs(t_clean - t_tail))

    delta_full = np.array(delta_full)
    delta_topk = np.array(delta_topk)

    # Mean and 95% CI (bootstrap or normal approximation)
    def mean_ci(x):
        m = x.mean()
        se = x.std(ddof=1) / np.sqrt(len(x))
        return m, m - 1.96 * se, m + 1.96 * se

    mf, lf, uf = mean_ci(delta_full)
    mt, lt, ut = mean_ci(delta_topk)

    return {
        "C": C,
        "tail_noise": tail_noise,
        "Full_ITP_mean": mf,
        "Full_ITP_CI_low": lf,
        "Full_ITP_CI_high": uf,
        "Topk_ITP_mean": mt,
        "Topk_ITP_CI_low": lt,
        "Topk_ITP_CI_high": ut,
        "Reduction_ratio": mf / (mt + 1e-12),  # how much Top-k reduces sensitivity
    }


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    print("=" * 90)
    print("CONTROLLED MONTE CARLO: Sensitivity of ITP to Tail Mass Accumulation")
    print("=" * 90)
    print(f"Top-k = {TOPK}, delta = {DELTA}, draws per cell = {N_DRAWS}")
    print(f"C values: {C_LIST}")
    print(f"Tail noise levels: {TAIL_NOISE_LEVELS}")
    print("=" * 90)

    rows = []
    for C in C_LIST:
        for noise in TAIL_NOISE_LEVELS:
            res = run_monte_carlo(C, noise, N_DRAWS, TOPK, DELTA)
            rows.append(res)
            print(
                f"C={C:<4} noise={noise:<5} | "
                f"Full ITP Δ={res['Full_ITP_mean']:.4f} "
                f"[{res['Full_ITP_CI_low']:.4f}, {res['Full_ITP_CI_high']:.4f}] | "
                f"Top-k ITP Δ={res['Topk_ITP_mean']:.4f} "
                f"[{res['Topk_ITP_CI_low']:.4f}, {res['Topk_ITP_CI_high']:.4f}] | "
                f"Ratio={res['Reduction_ratio']:.2f}x"
            )

    df = pd.DataFrame(rows)

    print("\n" + "=" * 90)
    print("SUMMARY TABLE (for manuscript/rebuttal)")
    print("=" * 90)
    print(
        df.to_string(
            index=False,
            float_format=lambda x: f"{x:.5f}",
        )
    )

    print("\n" + "=" * 90)
    print("INTERPRETATION")
    print("=" * 90)
    print(
        "If Top-k ITP shows a smaller Delta_ITP than Full-support ITP,\n"
        "it means Top-k is more stable under tail accumulation.\n"
        "This directly validates the design motivation: Top-k isolates\n"
        "decision-relevant dominant mass from irrelevant tail mass."
    )

CONTROLLED MONTE CARLO: Sensitivity of ITP to Tail Mass Accumulation
Top-k = 2, delta = 0.01, draws per cell = 10000
C values: [3, 10, 20, 50, 100]
Tail noise levels: [0.0, 0.05, 0.1, 0.2]
C=3    noise=0.0   | Full ITP Δ=0.0000 [0.0000, 0.0000] | Top-k ITP Δ=0.0000 [0.0000, 0.0000] | Ratio=0.00x
C=3    noise=0.05  | Full ITP Δ=0.0000 [0.0000, 0.0000] | Top-k ITP Δ=0.0000 [0.0000, 0.0000] | Ratio=0.00x
C=3    noise=0.1   | Full ITP Δ=0.0000 [0.0000, 0.0000] | Top-k ITP Δ=0.0000 [0.0000, 0.0000] | Ratio=0.00x
C=3    noise=0.2   | Full ITP Δ=0.0000 [0.0000, 0.0000] | Top-k ITP Δ=0.0000 [0.0000, 0.0000] | Ratio=0.00x
C=10   noise=0.0   | Full ITP Δ=0.0000 [0.0000, 0.0000] | Top-k ITP Δ=0.0000 [0.0000, 0.0000] | Ratio=0.00x
C=10   noise=0.05  | Full ITP Δ=0.0004 [0.0004, 0.0004] | Top-k ITP Δ=0.0003 [0.0003, 0.0003] | Ratio=1.24x
C=10   noise=0.1   | Full ITP Δ=0.0008 [0.0008, 0.0009] | Top-k ITP Δ=0.0006 [0.0005, 0.0006] | Ratio=1.49x
C=10   noise=0.2   | Full ITP Δ=0.0016 [0.0016, 0.0017]

# Top-k ITP under Support-Preserving and Support-Changing Perturbations

In [8]:
"""
standalone_topk_mechanism_montecarlo.py

Controlled Monte Carlo mechanism analysis for DPF Top-k ITP.

Purpose
-------
Test the specific mechanistic role of Top-k support control in
larger class spaces.

Two perturbation regimes are evaluated:

1. SUPPORT-PRESERVING TAIL REDISTRIBUTION
   - The true dominant Top-k probabilities are fixed.
   - Only the remaining C-k tail probabilities are redistributed.
   - Tail classes are constrained to remain below the kth dominant
     probability.
   - Question:
       How sensitive are Full-support ITP and Top-k ITP to changes
       confined to non-dominant predictive mass?

2. SUPPORT-CHANGING PERTURBATION
   - Probability mass is transferred from the kth dominant class
     to one tail class.
   - At sufficient perturbation, the tail class can enter Top-k.
   - Question:
       Does Top-k remain stable while dominant support is preserved,
       but respond once the dominant support actually changes?

This is a mechanism analysis, NOT a substitute for real-world
large-class multimodal evaluation.

No files are saved.
"""

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

SEED = 2027
N_DRAWS = 10_000

TOPK = 2
DELTA = 1e-2

# C=3 omitted:
# with k=2 there is only one tail class, so tail redistribution
# has no degrees of freedom.
C_LIST = [10, 20, 50, 100]

# Controls unevenness of tail redistribution.
# Larger values -> stronger deviation from uniform tail.
TAIL_PERTURBATION_LEVELS = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]

# Fraction of probability transferred from the kth dominant
# class toward one selected tail class.
SUPPORT_SHIFT_LEVELS = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]

rng = np.random.default_rng(SEED)


# ============================================================
# ITP definitions
# ============================================================

def itp_full_support(P, delta=1e-2):
    """
    Full-support ITP:
        -log(sum_c p_c^2 + delta * max_c p_c^2)
    """
    P = np.asarray(P, dtype=float)

    sum_sq = np.sum(P ** 2)
    max_sq = np.max(P ** 2)

    return -np.log(
        sum_sq
        + delta * max_sq
        + 1e-12
    )


def itp_topk(P, topk=2, delta=1e-2):
    """
    Practical Top-k ITP:
        -log(
            sum_{c in Top-k} p_c^2
            + delta * max_{c in Top-k} p_c^2
        )
    """
    P = np.asarray(P, dtype=float)

    k = min(topk, len(P))

    idx = np.argpartition(P, -k)[-k:]
    top = P[idx]

    sum_sq = np.sum(top ** 2)
    max_sq = np.max(top ** 2)

    return -np.log(
        sum_sq
        + delta * max_sq
        + 1e-12
    )


def topk_indices(P, k=2):
    """
    Return Top-k class indices.
    """
    P = np.asarray(P)
    k = min(k, len(P))

    return set(
        np.argpartition(P, -k)[-k:]
    )


# ============================================================
# Base distribution generator
# ============================================================

def generate_base_distribution(C, rng):
    """
    Generate a distribution with a guaranteed dominant Top-2.

    p1 is strongly dominant.
    p2 is second dominant.
    Remaining probability mass is uniformly distributed across
    C-2 tail classes.

    Construction guarantees:
        p1 > p2 > every tail probability.

    Returns
    -------
    P : ndarray, shape (C,)
    """

    if C < 3:
        raise ValueError("C must be >= 3.")

    # Dominant class
    p1 = rng.uniform(0.45, 0.65)

    # Second dominant class
    p2 = rng.uniform(0.15, 0.25)

    # Ensure enough probability remains
    if p1 + p2 >= 0.95:
        p2 = 0.95 - p1

    remaining = 1.0 - p1 - p2

    tail_n = C - 2
    tail_value = remaining / tail_n

    # Safety check:
    # second dominant probability must exceed all tail probabilities
    if tail_value >= p2:
        raise RuntimeError(
            f"Invalid construction: tail={tail_value:.4f} >= p2={p2:.4f}"
        )

    P = np.concatenate([
        np.array([p1, p2]),
        np.full(tail_n, tail_value)
    ])

    P /= P.sum()

    return P


# ============================================================
# Regime 1:
# Support-preserving tail redistribution
# ============================================================

def perturb_tail_support_preserving(
    P,
    perturbation_level,
    rng,
    topk=2
):
    """
    Redistribute ONLY tail mass while preserving the original
    Top-k support.

    The total tail mass remains fixed.

    perturbation_level:
        0.0 -> uniform tail (no change)
        1.0 -> strongly non-uniform tail

    We use a convex mixture between:
        uniform tail
        random Dirichlet tail

    Any candidate causing a tail probability to exceed the kth
    dominant probability is rejected and regenerated.

    Returns
    -------
    P_new
    """

    P = np.asarray(P, dtype=float)

    dominant = P[:topk].copy()
    tail = P[topk:].copy()

    tail_n = len(tail)
    tail_total = tail.sum()

    if tail_n <= 1 or perturbation_level == 0:
        return P.copy()

    kth_dominant = np.min(dominant)

    uniform_tail = np.full(
        tail_n,
        tail_total / tail_n
    )

    # Try multiple times to obtain a support-preserving tail
    for _ in range(100):

        # Low alpha -> more uneven random tail
        random_tail = rng.dirichlet(
            np.ones(tail_n) * 0.5
        ) * tail_total

        mixed_tail = (
            (1.0 - perturbation_level) * uniform_tail
            + perturbation_level * random_tail
        )

        # Strict support preservation
        if np.max(mixed_tail) < kth_dominant:
            P_new = np.concatenate([
                dominant,
                mixed_tail
            ])

            P_new /= P_new.sum()

            return P_new

    # If an extreme random draw repeatedly violates support,
    # fall back to the strongest valid mixture via shrinkage.
    random_tail = rng.dirichlet(
        np.ones(tail_n) * 0.5
    ) * tail_total

    lam = perturbation_level

    while lam > 1e-6:

        mixed_tail = (
            (1.0 - lam) * uniform_tail
            + lam * random_tail
        )

        if np.max(mixed_tail) < kth_dominant:
            P_new = np.concatenate([
                dominant,
                mixed_tail
            ])

            P_new /= P_new.sum()

            return P_new

        lam *= 0.5

    return P.copy()


# ============================================================
# Regime 2:
# Support-changing perturbation
# ============================================================

def perturb_support_changing(
    P,
    shift_level,
    rng,
    topk=2
):
    """
    Transfer probability mass from the kth dominant class
    to one randomly selected tail class.

    shift_level controls progress toward / beyond the support
    crossover point.

    The transfer amount is defined relative to the amount needed
    for the selected tail class to equal the kth dominant class.

    shift_level:
        0.00 -> no change
        0.50 -> halfway toward crossover
        1.00 -> slightly beyond crossover

    This permits direct measurement of whether the Top-k support
    has changed.

    Returns
    -------
    P_new
    """

    P = np.asarray(P, dtype=float).copy()

    if shift_level == 0:
        return P

    kth_idx = topk - 1

    tail_indices = np.arange(
        topk,
        len(P)
    )

    target_idx = rng.choice(
        tail_indices
    )

    p_k = P[kth_idx]
    p_tail = P[target_idx]

    # If x is transferred:
    #
    # new kth = p_k - x
    # new tail = p_tail + x
    #
    # crossover:
    # p_tail + x = p_k - x
    #
    # x_cross = (p_k - p_tail) / 2

    x_cross = (
        p_k - p_tail
    ) / 2.0

    # At shift_level = 1:
    # move slightly beyond crossover.
    #
    # Multiplying by 1.10 ensures actual support replacement.
    transfer = (
        shift_level
        * 1.10
        * x_cross
    )

    # Never transfer more than available probability
    transfer = min(
        transfer,
        P[kth_idx] - 1e-8
    )

    P[kth_idx] -= transfer
    P[target_idx] += transfer

    P = np.clip(
        P,
        0,
        None
    )

    P /= P.sum()

    return P


# ============================================================
# Confidence interval helper
# ============================================================

def mean_ci(x):
    """
    Mean and normal-approximation 95% CI.
    With n=10,000 this is adequate for descriptive Monte Carlo
    summaries.
    """
    x = np.asarray(x, dtype=float)

    mean = np.mean(x)
    sd = np.std(x, ddof=1)
    se = sd / np.sqrt(len(x))

    low = mean - 1.96 * se
    high = mean + 1.96 * se

    return mean, low, high


# ============================================================
# Monte Carlo:
# Support-preserving regime
# ============================================================

def run_support_preserving(
    C,
    perturbation_level,
    n_draws,
    rng
):
    """
    Measure potential sensitivity when ONLY non-dominant tail
    structure changes and Top-k support remains fixed.
    """

    delta_full = np.empty(n_draws)
    delta_topk = np.empty(n_draws)

    support_changes = 0

    for i in range(n_draws):

        P_base = generate_base_distribution(
            C,
            rng
        )

        base_support = topk_indices(
            P_base,
            TOPK
        )

        P_perturbed = perturb_tail_support_preserving(
            P_base,
            perturbation_level,
            rng,
            TOPK
        )

        perturbed_support = topk_indices(
            P_perturbed,
            TOPK
        )

        if base_support != perturbed_support:
            support_changes += 1

        full_before = itp_full_support(
            P_base,
            DELTA
        )

        full_after = itp_full_support(
            P_perturbed,
            DELTA
        )

        topk_before = itp_topk(
            P_base,
            TOPK,
            DELTA
        )

        topk_after = itp_topk(
            P_perturbed,
            TOPK,
            DELTA
        )

        delta_full[i] = abs(
            full_after - full_before
        )

        delta_topk[i] = abs(
            topk_after - topk_before
        )

    mf, lf, uf = mean_ci(delta_full)
    mt, lt, ut = mean_ci(delta_topk)

    return {
        "C": C,
        "Perturbation": perturbation_level,

        "Full_mean": mf,
        "Full_CI_low": lf,
        "Full_CI_high": uf,

        "Topk_mean": mt,
        "Topk_CI_low": lt,
        "Topk_CI_high": ut,

        "Support_change_rate":
            support_changes / n_draws,

        "Sensitivity_reduction":
            mf / (mt + 1e-12)
    }


# ============================================================
# Monte Carlo:
# Support-changing regime
# ============================================================

def run_support_changing(
    C,
    shift_level,
    n_draws,
    rng
):
    """
    Measure potential response as perturbation approaches and
    crosses the Top-k support boundary.
    """

    delta_full = np.empty(n_draws)
    delta_topk = np.empty(n_draws)

    support_changes = 0

    for i in range(n_draws):

        P_base = generate_base_distribution(
            C,
            rng
        )

        base_support = topk_indices(
            P_base,
            TOPK
        )

        P_shifted = perturb_support_changing(
            P_base,
            shift_level,
            rng,
            TOPK
        )

        shifted_support = topk_indices(
            P_shifted,
            TOPK
        )

        if base_support != shifted_support:
            support_changes += 1

        full_before = itp_full_support(
            P_base,
            DELTA
        )

        full_after = itp_full_support(
            P_shifted,
            DELTA
        )

        topk_before = itp_topk(
            P_base,
            TOPK,
            DELTA
        )

        topk_after = itp_topk(
            P_shifted,
            TOPK,
            DELTA
        )

        delta_full[i] = abs(
            full_after - full_before
        )

        delta_topk[i] = abs(
            topk_after - topk_before
        )

    mf, lf, uf = mean_ci(delta_full)
    mt, lt, ut = mean_ci(delta_topk)

    return {
        "C": C,
        "Shift": shift_level,

        "Full_mean": mf,
        "Full_CI_low": lf,
        "Full_CI_high": uf,

        "Topk_mean": mt,
        "Topk_CI_low": lt,
        "Topk_CI_high": ut,

        "Support_change_rate":
            support_changes / n_draws
    }


# ============================================================
# Sanity checks
# ============================================================

def run_sanity_checks():

    print("=" * 90)
    print("SANITY CHECKS")
    print("=" * 90)

    local_rng = np.random.default_rng(
        SEED + 999
    )

    for C in C_LIST:

        P = generate_base_distribution(
            C,
            local_rng
        )

        sorted_p = np.sort(P)[::-1]

        assert sorted_p[0] == P[0]
        assert sorted_p[1] == P[1]

        assert np.max(
            P[TOPK:]
        ) < P[TOPK - 1]

        P_preserve = perturb_tail_support_preserving(
            P,
            1.0,
            local_rng,
            TOPK
        )

        assert (
            topk_indices(P, TOPK)
            ==
            topk_indices(P_preserve, TOPK)
        )

        topk_before = itp_topk(
            P,
            TOPK,
            DELTA
        )

        topk_after = itp_topk(
            P_preserve,
            TOPK,
            DELTA
        )

        # Because dominant probabilities are exactly fixed,
        # Top-k ITP should be numerically invariant.
        assert np.isclose(
            topk_before,
            topk_after,
            atol=1e-12
        )

        print(
            f"C={C:<4} PASS | "
            f"p1={P[0]:.3f}, "
            f"p2={P[1]:.3f}, "
            f"max-tail={np.max(P[2:]):.4f}"
        )

    print("All sanity checks passed.\n")


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    run_sanity_checks()

    # --------------------------------------------------------
    # Regime 1
    # --------------------------------------------------------

    print("=" * 90)
    print("REGIME 1: SUPPORT-PRESERVING TAIL REDISTRIBUTION")
    print("=" * 90)
    print(
        "Dominant Top-k probabilities are fixed; "
        "only non-dominant tail structure changes."
    )
    print()

    preserving_rows = []

    for C in C_LIST:

        for level in TAIL_PERTURBATION_LEVELS:

            result = run_support_preserving(
                C,
                level,
                N_DRAWS,
                rng
            )

            preserving_rows.append(
                result
            )

            print(
                f"C={C:<4} "
                f"perturb={level:<4.2f} | "
                f"Full Δ={result['Full_mean']:.6f} | "
                f"Top-k Δ={result['Topk_mean']:.6f} | "
                f"support-change="
                f"{result['Support_change_rate']:.4f}"
            )

    df_preserve = pd.DataFrame(
        preserving_rows
    )

    print("\n" + "=" * 90)
    print("SUPPORT-PRESERVING SUMMARY")
    print("=" * 90)

    print(
        df_preserve.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


    # --------------------------------------------------------
    # Regime 2
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("REGIME 2: SUPPORT-CHANGING PERTURBATION")
    print("=" * 90)
    print(
        "Probability is transferred from the kth dominant class "
        "to one tail class."
    )
    print()

    changing_rows = []

    for C in C_LIST:

        for level in SUPPORT_SHIFT_LEVELS:

            result = run_support_changing(
                C,
                level,
                N_DRAWS,
                rng
            )

            changing_rows.append(
                result
            )

            print(
                f"C={C:<4} "
                f"shift={level:<4.2f} | "
                f"Full Δ={result['Full_mean']:.6f} | "
                f"Top-k Δ={result['Topk_mean']:.6f} | "
                f"support-change="
                f"{result['Support_change_rate']:.4f}"
            )

    df_change = pd.DataFrame(
        changing_rows
    )

    print("\n" + "=" * 90)
    print("SUPPORT-CHANGING SUMMARY")
    print("=" * 90)

    print(
        df_change.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


    # --------------------------------------------------------
    # Compact rebuttal summary
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("COMPACT REBUTTAL SUMMARY")
    print("=" * 90)

    print("\nSupport-preserving regime:")
    print(
        "Top-k ITP should remain invariant while full-support "
        "ITP responds to changes confined to non-dominant tail structure."
    )

    for C in C_LIST:

        row = df_preserve[
            (df_preserve["C"] == C)
            &
            (
                df_preserve["Perturbation"]
                ==
                max(TAIL_PERTURBATION_LEVELS)
            )
        ].iloc[0]

        print(
            f"  C={C:<4}: "
            f"Full Δ={row['Full_mean']:.6f}, "
            f"Top-k Δ={row['Topk_mean']:.6f}, "
            f"support-change="
            f"{row['Support_change_rate']:.4f}"
        )

    print("\nSupport-changing regime:")

    for C in C_LIST:

        row = df_change[
            (df_change["C"] == C)
            &
            (
                df_change["Shift"]
                ==
                max(SUPPORT_SHIFT_LEVELS)
            )
        ].iloc[0]

        print(
            f"  C={C:<4}: "
            f"Full Δ={row['Full_mean']:.6f}, "
            f"Top-k Δ={row['Topk_mean']:.6f}, "
            f"support-change="
            f"{row['Support_change_rate']:.4f}"
        )

    print("\nInterpretation:")
    print(
        "Regime 1 tests selective invariance: Top-k ignores changes "
        "confined to non-dominant support."
    )
    print(
        "Regime 2 tests responsiveness: Top-k changes once perturbation "
        "alters the dominant support."
    )
    print(
        "These simulations isolate the support-control mechanism; "
        "they do not establish real-world large-class classification performance."
    )

SANITY CHECKS
C=10   PASS | p1=0.594, p2=0.157, max-tail=0.0312
C=20   PASS | p1=0.597, p2=0.229, max-tail=0.0097
C=50   PASS | p1=0.515, p2=0.186, max-tail=0.0062
C=100  PASS | p1=0.472, p2=0.183, max-tail=0.0035
All sanity checks passed.

REGIME 1: SUPPORT-PRESERVING TAIL REDISTRIBUTION
Dominant Top-k probabilities are fixed; only non-dominant tail structure changes.

C=10   perturb=0.00 | Full Δ=0.000000 | Top-k Δ=0.000000 | support-change=0.0000
C=10   perturb=0.25 | Full Δ=0.002258 | Top-k Δ=0.000000 | support-change=0.0000
C=10   perturb=0.50 | Full Δ=0.009005 | Top-k Δ=0.000000 | support-change=0.0000
C=10   perturb=0.75 | Full Δ=0.019077 | Top-k Δ=0.000000 | support-change=0.0000
C=10   perturb=1.00 | Full Δ=0.030634 | Top-k Δ=0.000000 | support-change=0.0000
C=20   perturb=0.00 | Full Δ=0.000000 | Top-k Δ=0.000000 | support-change=0.0000
C=20   perturb=0.25 | Full Δ=0.001241 | Top-k Δ=0.000000 | support-change=0.0000
C=20   perturb=0.50 | Full Δ=0.004978 | Top-k Δ=0.000000 | s

# Fixed-β interpretation ablation

In [9]:
"""
Controlled interpretation ablation for DPF.

Purpose:
    Hold the Gibbs realization parameter beta FIXED and vary only
    the interpretation functional:

        1. Shannon entropy
        2. Renyi-2
        3. Practical ITP

    This isolates the interpretation functional from beta selection.

Protocol:
    - Common beta = 1.5 for ALL three interpretations
    - Same precomputed upstream probabilities
    - Same Gibbs mapping
    - Same classifier architecture
    - 30 paired runs
    - 2x stratified bootstrap training samples
    - Same bootstrap indices within each run for every method

No results are saved.
"""

import pickle
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import f1_score
from sklearn.utils import resample
from scipy.stats import ttest_rel, wilcoxon, t


# ============================================================
# Configuration
# ============================================================

PKL_PATH = "../cache/dpf_mvsa_single_results.pkl"

FIXED_BETA = 1.5

TOPK = 2
DELTA = 1e-2

N_RUNS = 30
BASE_SEED = 42
BOOTSTRAP_FACTOR = 2


# ============================================================
# Load probabilities
# ============================================================

print("=" * 80)
print("LOADING PRECOMPUTED PROBABILITIES")
print("=" * 80)

with open(PKL_PATH, "rb") as f:
    results = pickle.load(f)

train_probs_text = results["train_probs_text"]
train_probs_image = results["train_probs_image"]

test_probs_text = results["test_probs_text"]
test_probs_image = results["test_probs_image"]

y_train = results["y_train_int"]
y_test = results["y_test_int"]

print("Train:", train_probs_text.shape, train_probs_image.shape)
print("Test: ", test_probs_text.shape, test_probs_image.shape)
print(f"Fixed beta = {FIXED_BETA}")
print(f"Runs = {N_RUNS}")
print(f"Bootstrap factor = {BOOTSTRAP_FACTOR}x")


# ============================================================
# Interpretation functionals
# ============================================================

def compute_potential(probs, method, topk=2, delta=1e-2):

    probs = np.asarray(probs)

    if method == "shannon":

        return -np.sum(
            probs * np.log(probs + 1e-12),
            axis=1
        )

    elif method == "renyi2":

        return -np.log(
            np.sum(probs ** 2, axis=1)
            + 1e-12
        )

    elif method == "itp":

        k = min(topk, probs.shape[1])

        idx = np.argpartition(
            probs,
            -k,
            axis=1
        )[:, -k:]

        topk_probs = np.take_along_axis(
            probs,
            idx,
            axis=1
        )

        sum_topk_sq = np.sum(
            topk_probs ** 2,
            axis=1
        )

        max_topk_sq = np.max(
            topk_probs ** 2,
            axis=1
        )

        return -np.log(
            sum_topk_sq
            + delta * max_topk_sq
            + 1e-12
        )

    else:
        raise ValueError(
            f"Unknown method: {method}"
        )


# ============================================================
# Fixed-beta Gibbs realization
# ============================================================

def gibbs_fusion(
    probs_text,
    probs_image,
    method,
    beta=FIXED_BETA,
    topk=TOPK,
    delta=DELTA
):

    e_text = compute_potential(
        probs_text,
        method,
        topk,
        delta
    )

    e_image = compute_potential(
        probs_image,
        method,
        topk,
        delta
    )

    energies = np.column_stack([
        e_text,
        e_image
    ])

    # Numerically stable Gibbs softmin
    logits = -beta * energies

    logits -= np.max(
        logits,
        axis=1,
        keepdims=True
    )

    weights = np.exp(logits)

    weights /= np.sum(
        weights,
        axis=1,
        keepdims=True
    )

    fused_text = (
        weights[:, 0:1]
        * probs_text
    )

    fused_image = (
        weights[:, 1:2]
        * probs_image
    )

    return np.concatenate(
        [
            fused_text,
            fused_image
        ],
        axis=1
    )


# ============================================================
# Precompute fused representations
# ============================================================

METHODS = [
    ("Shannon", "shannon"),
    ("Renyi-2", "renyi2"),
    ("Practical ITP", "itp"),
]

print("\n" + "=" * 80)
print("PRECOMPUTING FIXED-BETA REPRESENTATIONS")
print("=" * 80)

fused_train = {}
fused_test = {}

for display_name, method in METHODS:

    fused_train[method] = gibbs_fusion(
        train_probs_text,
        train_probs_image,
        method
    )

    fused_test[method] = gibbs_fusion(
        test_probs_text,
        test_probs_image,
        method
    )

    print(
        f"{display_name:<15} "
        f"beta={FIXED_BETA}"
    )


# ============================================================
# Stratified 2x bootstrap
# ============================================================

def stratified_bootstrap_indices(
    y,
    factor,
    seed
):

    rng = np.random.default_rng(seed)

    indices = []

    classes = np.unique(y)

    for cls in classes:

        cls_idx = np.where(
            y == cls
        )[0]

        n_sample = (
            len(cls_idx)
            * factor
        )

        sampled = rng.choice(
            cls_idx,
            size=n_sample,
            replace=True
        )

        indices.extend(sampled)

    indices = np.asarray(indices)

    rng.shuffle(indices)

    return indices


# ============================================================
# 30 paired runs
# ============================================================

print("\n" + "=" * 80)
print(
    f"30 PAIRED RUNS — FIXED beta={FIXED_BETA}"
)
print("=" * 80)

scores = {
    method: []
    for _, method in METHODS
}

for run in range(N_RUNS):

    seed = BASE_SEED + run

    # SAME bootstrap sample for every interpretation
    boot_idx = stratified_bootstrap_indices(
        y_train,
        BOOTSTRAP_FACTOR,
        seed
    )

    print(
        f"\nRun {run + 1:02d}/{N_RUNS} "
        f"| seed={seed} "
        f"| bootstrap N={len(boot_idx)}"
    )

    for display_name, method in METHODS:

        X_train_boot = (
            fused_train[method][boot_idx]
        )

        y_train_boot = (
            y_train[boot_idx]
        )

        clf = xgb.XGBClassifier(
            n_estimators=350,
            max_depth=5,
            random_state=seed,
            eval_metric="mlogloss",
            tree_method="hist",
        )

        clf.fit(
            X_train_boot,
            y_train_boot
        )

        pred = clf.predict(
            fused_test[method]
        )

        wf1 = f1_score(
            y_test,
            pred,
            average="weighted"
        )

        scores[method].append(
            wf1
        )

        print(
            f"  {display_name:<15} "
            f"W-F1={wf1:.4f}"
        )


# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 80)
print("FIXED-BETA 30-RUN SUMMARY")
print("=" * 80)

summary_rows = []

for display_name, method in METHODS:

    x = np.asarray(
        scores[method]
    )

    summary_rows.append({
        "Interpretation": display_name,
        "beta": FIXED_BETA,
        "Mean_WF1": np.mean(x),
        "SD_WF1": np.std(
            x,
            ddof=1
        ),
        "Min_WF1": np.min(x),
        "Max_WF1": np.max(x),
    })

summary_df = pd.DataFrame(
    summary_rows
)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# Pairwise statistical comparisons
# ============================================================

def paired_statistics(a, b):

    a = np.asarray(a)
    b = np.asarray(b)

    diff = a - b

    mean_diff = np.mean(diff)
    sd_diff = np.std(
        diff,
        ddof=1
    )

    se = (
        sd_diff
        / np.sqrt(len(diff))
    )

    tcrit = t.ppf(
        0.975,
        df=len(diff) - 1
    )

    ci_low = (
        mean_diff
        - tcrit * se
    )

    ci_high = (
        mean_diff
        + tcrit * se
    )

    t_stat, t_p = ttest_rel(
        a,
        b
    )

    try:
        w_stat, w_p = wilcoxon(
            a,
            b,
            zero_method="wilcox"
        )
    except ValueError:
        w_stat = np.nan
        w_p = np.nan

    return {
        "Mean_Difference": mean_diff,
        "SD_Difference": sd_diff,
        "CI95_Low": ci_low,
        "CI95_High": ci_high,
        "Paired_t": t_stat,
        "Paired_t_p": t_p,
        "Wilcoxon_W": w_stat,
        "Wilcoxon_p": w_p,
    }


print("\n" + "=" * 80)
print("PAIRED INTERPRETATION COMPARISONS")
print("=" * 80)

comparisons = [
    (
        "ITP - Shannon",
        "itp",
        "shannon"
    ),
    (
        "ITP - Renyi-2",
        "itp",
        "renyi2"
    ),
    (
        "Renyi-2 - Shannon",
        "renyi2",
        "shannon"
    ),
]

comparison_rows = []

for name, a, b in comparisons:

    stats = paired_statistics(
        scores[a],
        scores[b]
    )

    comparison_rows.append({
        "Comparison": name,
        **stats
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# Compact rebuttal output
# ============================================================

print("\n" + "=" * 80)
print("COMPACT REBUTTAL OUTPUT")
print("=" * 80)

for display_name, method in METHODS:

    x = np.asarray(
        scores[method]
    )

    print(
        f"{display_name:<15} "
        f"beta={FIXED_BETA:<4} | "
        f"W-F1="
        f"{np.mean(x):.4f} "
        f"± {np.std(x, ddof=1):.4f}"
    )

print(
    "\nInterpretation-only paired comparisons "
    f"(common beta={FIXED_BETA}):"
)

for row in comparison_rows:

    print(
        f"  {row['Comparison']:<20} "
        f"Δ={row['Mean_Difference']:+.4f}, "
        f"95% CI "
        f"[{row['CI95_Low']:+.4f}, "
        f"{row['CI95_High']:+.4f}], "
        f"paired-t p="
        f"{row['Paired_t_p']:.3e}, "
        f"Wilcoxon p="
        f"{row['Wilcoxon_p']:.3e}"
    )

LOADING PRECOMPUTED PROBABILITIES
Train: (3895, 3) (3895, 3)
Test:  (487, 3) (487, 3)
Fixed beta = 1.5
Runs = 30
Bootstrap factor = 2x

PRECOMPUTING FIXED-BETA REPRESENTATIONS
Shannon         beta=1.5
Renyi-2         beta=1.5
Practical ITP   beta=1.5

30 PAIRED RUNS — FIXED beta=1.5

Run 01/30 | seed=42 | bootstrap N=7790
  Shannon         W-F1=0.5007
  Renyi-2         W-F1=0.5162
  Practical ITP   W-F1=0.5250

Run 02/30 | seed=43 | bootstrap N=7790
  Shannon         W-F1=0.5141
  Renyi-2         W-F1=0.5170
  Practical ITP   W-F1=0.5383

Run 03/30 | seed=44 | bootstrap N=7790
  Shannon         W-F1=0.5132
  Renyi-2         W-F1=0.5172
  Practical ITP   W-F1=0.5151

Run 04/30 | seed=45 | bootstrap N=7790
  Shannon         W-F1=0.5170
  Renyi-2         W-F1=0.5150
  Practical ITP   W-F1=0.5250

Run 05/30 | seed=46 | bootstrap N=7790
  Shannon         W-F1=0.5210
  Renyi-2         W-F1=0.5433
  Practical ITP   W-F1=0.5369

Run 06/30 | seed=47 | bootstrap N=7790
  Shannon         W-F1=0.5

# misclassification analysis

In [2]:
"""
R3 W1: Concentration-reliability analysis on misclassified samples.
Uses the same validation probabilities and Practical ITP definition
as the DPF experiments.
"""

import pickle
import numpy as np
from scipy.stats import binomtest

PKL_PATH = "../cache/dpf_mvsa_single_results.pkl"
TOPK = 2
DELTA = 1e-2

with open(PKL_PATH, "rb") as f:
    results = pickle.load(f)

val_probs_text  = np.asarray(results["val_probs_text"])
val_probs_image = np.asarray(results["val_probs_image"])
y_val           = np.asarray(results["y_val_int"])

def compute_itp(probs, topk=TOPK, delta=DELTA):
    k = min(topk, probs.shape[1])
    topk_probs = np.sort(probs, axis=1)[:, -k:]
    concentration = np.sum(topk_probs ** 2, axis=1)
    regularizer   = delta * np.max(topk_probs ** 2, axis=1)
    return -np.log(concentration + regularizer + 1e-12)

itp_text = compute_itp(val_probs_text)
itp_img  = compute_itp(val_probs_image)

pred_text = np.argmax(val_probs_text, axis=1)
pred_img  = np.argmax(val_probs_image, axis=1)

text_correct = pred_text == y_val
img_correct  = pred_img == y_val

# Samples for which exactly one modality is correct
one_correct = text_correct ^ img_correct

gap = itp_text - itp_img
tie = np.isclose(gap, 0.0, atol=1e-12)

mask = one_correct & ~tie

lower_itp_is_text = itp_text < itp_img
lower_itp_correct = np.where(
    lower_itp_is_text,
    text_correct,
    img_correct
)

n_total = len(y_val)
n_one   = int(one_correct.sum())
n_tie   = int((one_correct & tie).sum())
n_used  = int(mask.sum())
n_good  = int(lower_itp_correct[mask].sum())

correct_rate = n_good / n_used
wrong_rate   = 1.0 - correct_rate

# Exact binomial test against 50% ordering accuracy
binom = binomtest(n_good, n_used, p=0.5, alternative="two-sided")

print("=" * 65)
print("R3 W1: Concentration-reliability on one-correct samples")
print("=" * 65)
print(f"Total validation samples:          {n_total}")
print(f"Exactly-one-correct samples:       {n_one}")
print(f"Ties:                              {n_tie}")
print(f"Samples used:                      {n_used}")
print(f"Lower-ITP modality correct:        {correct_rate:.2%} ({n_good}/{n_used})")
print(f"Wrong modality more concentrated: {wrong_rate:.2%} ({n_used-n_good}/{n_used})")
print(f"Exact binomial test vs. 50%:       p={binom.pvalue:.4f}")

R3 W1: Concentration-reliability on one-correct samples
Total validation samples:          487
Exactly-one-correct samples:       101
Ties:                              0
Samples used:                      101
Lower-ITP modality correct:        57.43% (58/101)
Wrong modality more concentrated: 42.57% (43/101)
Exact binomial test vs. 50%:       p=0.1633


# R3 W2: Max-confidence selection + XGBoost downstream.

In [16]:
"""
Pipeline matches Table 5:
    - Same precomputed upstream probabilities
    - Same XGBoost architecture
    - Same 30-run paired stratified bootstrap
    - Same bootstrap indices across methods within each run

Representation:
    confidence-proportional weighting + concatenation:
        conf_text  = max(p_text)
        conf_image = max(p_image)
        w_text  = conf_text / (conf_text + conf_image)
        w_image = conf_image / (conf_text + conf_image)
        fused   = [w_text * p_text, w_image * p_image]
    then XGBoost downstream.

This mirrors DPF's concatenation structure but replaces ITP+Gibbs
weights with max-confidence weights.
"""

import pickle
import numpy as np
import xgboost as xgb

from sklearn.metrics import f1_score

# ============================================================
# Config
# ============================================================

PKL_PATH = "../cache/dpf_mvsa_single_results.pkl"

N_RUNS = 30
BASE_SEED = 42
BOOTSTRAP_FACTOR = 2.0


# ============================================================
# Load
# ============================================================

with open(PKL_PATH, "rb") as f:
    results = pickle.load(f)

train_probs_text  = np.asarray(results["train_probs_text"])
train_probs_image = np.asarray(results["train_probs_image"])
test_probs_text   = np.asarray(results["test_probs_text"])
test_probs_image  = np.asarray(results["test_probs_image"])

y_train = np.asarray(results["y_train_int"])
y_test  = np.asarray(results["y_test_int"])

print("=" * 80)
print("R3 W2: MAX-CONFIDENCE + XGBOOST DOWNSTREAM")
print("=" * 80)
print(f"Train: {train_probs_text.shape}, {train_probs_image.shape}")
print(f"Test:  {test_probs_text.shape}, {test_probs_image.shape}")
print(f"Runs:  {N_RUNS}")
print()


# ============================================================
# Max-confidence weighting + concatenation
# ============================================================

def confidence_weighted_concat(p_text, p_image):
    """
    Confidence-proportional weighting followed by concatenation.
    This mirrors DPF's concatenation structure, but replaces
    ITP+Gibbs weights with max-confidence weights.
    """
    conf_text  = p_text.max(axis=1, keepdims=True)
    conf_image = p_image.max(axis=1, keepdims=True)

    denom = conf_text + conf_image + 1e-12
    w_text  = conf_text / denom
    w_image = conf_image / denom

    return np.concatenate([
        w_text * p_text,
        w_image * p_image,
    ], axis=1)


# ============================================================
# XGBoost
# ============================================================

def make_classifier(seed):
    return xgb.XGBClassifier(
        n_estimators=350,
        max_depth=5,
        random_state=seed,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
    )


# ============================================================
# Stratified 2x bootstrap
# ============================================================

def stratified_bootstrap_indices(y, factor, seed):
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        n_sample = int(len(cls_idx) * factor)
        idx.extend(rng.choice(cls_idx, size=n_sample, replace=True))
    idx = np.asarray(idx)
    rng.shuffle(idx)
    return idx


# ============================================================
# Precompute representations (weights are not learned)
# ============================================================

X_train_mc = confidence_weighted_concat(train_probs_text, train_probs_image)
X_test_mc  = confidence_weighted_concat(test_probs_text, test_probs_image)

print("Representation shapes:")
print(f"  X_train_mc: {X_train_mc.shape}")
print(f"  X_test_mc:  {X_test_mc.shape}")
print()


# ============================================================
# 30 paired runs
# ============================================================

scores = []

for run in range(N_RUNS):
    seed = BASE_SEED + run
    idx = stratified_bootstrap_indices(y_train, BOOTSTRAP_FACTOR, seed)

    clf = make_classifier(seed)
    clf.fit(X_train_mc[idx], y_train[idx])

    pred = clf.predict(X_test_mc)
    f1 = f1_score(y_test, pred, average="weighted")

    scores.append(f1)

    print(f"Run {run + 1:02d}/{N_RUNS} | W-F1={f1:.4f}")

scores = np.asarray(scores)


# ============================================================
# Summary
# ============================================================

mean_score = scores.mean()
sd_score   = scores.std(ddof=1)

print("\n" + "=" * 80)
print("SUMMARY — MAX-CONFIDENCE + XGBOOST")
print("=" * 80)
print(f"Mean ± Std: {mean_score:.4f} ± {sd_score:.4f}")
print(f"Min / Max:  {scores.min():.4f} / {scores.max():.4f}")

R3 W2: MAX-CONFIDENCE + XGBOOST DOWNSTREAM
Train: (3895, 3), (3895, 3)
Test:  (487, 3), (487, 3)
Runs:  30

Representation shapes:
  X_train_mc: (3895, 6)
  X_test_mc:  (487, 6)

Run 01/30 | W-F1=0.5128
Run 02/30 | W-F1=0.5154
Run 03/30 | W-F1=0.5385
Run 04/30 | W-F1=0.5090
Run 05/30 | W-F1=0.5192
Run 06/30 | W-F1=0.5257
Run 07/30 | W-F1=0.5167
Run 08/30 | W-F1=0.5197
Run 09/30 | W-F1=0.5181
Run 10/30 | W-F1=0.5260
Run 11/30 | W-F1=0.5172
Run 12/30 | W-F1=0.5345
Run 13/30 | W-F1=0.5202
Run 14/30 | W-F1=0.5290
Run 15/30 | W-F1=0.5138
Run 16/30 | W-F1=0.5287
Run 17/30 | W-F1=0.5123
Run 18/30 | W-F1=0.5200
Run 19/30 | W-F1=0.5256
Run 20/30 | W-F1=0.4990
Run 21/30 | W-F1=0.5217
Run 22/30 | W-F1=0.5183
Run 23/30 | W-F1=0.5093
Run 24/30 | W-F1=0.5147
Run 25/30 | W-F1=0.5176
Run 26/30 | W-F1=0.5112
Run 27/30 | W-F1=0.5250
Run 28/30 | W-F1=0.5179
Run 29/30 | W-F1=0.5145
Run 30/30 | W-F1=0.5228

SUMMARY — MAX-CONFIDENCE + XGBOOST
Mean ± Std: 0.5191 ± 0.0080
Min / Max:  0.4990 / 0.5385


# R3 W4: Classifier-free mechanism probe for DPF weights.

In [5]:
"""
R3 W4: Classifier-free mechanism probe for DPF weights.

Purpose
-------
Test whether DPF's adaptive weights retain predictive utility without
a learned downstream classifier.

For both conditions:
    fused = w_text * p_text + w_image * p_image
    pred  = argmax(fused)

Uniform: w_text = w_image = 0.5
DPF:     weights from Practical ITP + Gibbs mapping

Protocol
--------
- Same precomputed MVSA-Single upstream probabilities.
- Practical-ITP beta fixed at the validation-selected value (beta=1.5);
  no test-set tuning.
- 30 paired stratified bootstrap evaluations.
- Identical bootstrap indices for DPF and uniform within each replicate.
- Report mean ± std weighted-F1, paired difference, 95% CI,
  and Wilcoxon signed-rank test.

This is a classifier-free isolation probe, not an alternative DPF
configuration. The original DPF uses concatenation + learned downstream.
"""

import pickle
import numpy as np

from sklearn.metrics import f1_score
from scipy.stats import wilcoxon, t


# ============================================================
# Configuration
# ============================================================

PKL_PATH = "../cache/dpf_mvsa_single_results.pkl"

TOPK = 2
DELTA = 1e-2

# Fixed to the validation-selected Practical-ITP beta from the
# 30-run Table 5 experiment; no beta tuning is performed on test data.
BETA_ITP = 1.5

N_RUNS = 30
BASE_SEED = 42

# Match Table 2's stratified bootstrap factor if Table 2 used
# test-set bootstrap. Set to 1.0 if only a paired evaluation
# (no resampling) is desired.
BOOTSTRAP_FACTOR = 2.0


# ============================================================
# Load precomputed probabilities
# ============================================================

with open(PKL_PATH, "rb") as f:
    results = pickle.load(f)

test_probs_text  = np.asarray(results["test_probs_text"])
test_probs_image = np.asarray(results["test_probs_image"])
y_test           = np.asarray(results["y_test_int"])

print("=" * 80)
print("R3 W4: CLASSIFIER-FREE MECHANISM PROBE")
print("=" * 80)
print(f"Test samples:       {len(y_test)}")
print(f"Fixed beta (ITP):   {BETA_ITP}")
print(f"Runs:               {N_RUNS}")
print(f"Bootstrap factor:   {BOOTSTRAP_FACTOR}")
print()


# ============================================================
# ITP
# ============================================================

def compute_itp(probs, topk=TOPK, delta=DELTA):
    probs = np.asarray(probs)
    k = min(topk, probs.shape[1])
    idx = np.argpartition(probs, -k, axis=1)[:, -k:]
    topk_probs = np.take_along_axis(probs, idx, axis=1)
    sum_topk_sq = np.sum(topk_probs ** 2, axis=1)
    max_topk_sq = np.max(topk_probs ** 2, axis=1)
    return -np.log(sum_topk_sq + delta * max_topk_sq + 1e-12)


# ============================================================
# Gibbs weights
# ============================================================

def gibbs_weights(probs_text, probs_image, beta, topk=TOPK, delta=DELTA):
    if beta == 0:
        n = len(probs_text)
        return np.full((n, 1), 0.5), np.full((n, 1), 0.5)

    e_text = compute_itp(probs_text, topk, delta)
    e_image = compute_itp(probs_image, topk, delta)

    energies = np.column_stack([e_text, e_image])
    logits = -beta * energies
    logits -= np.max(logits, axis=1, keepdims=True)

    weights = np.exp(logits)
    weights /= np.sum(weights, axis=1, keepdims=True)

    return weights[:, 0:1], weights[:, 1:2]


# ============================================================
# Weighted average + argmax (non-learned downstream)
# ============================================================

def weighted_average_predict(probs_text, probs_image, w_text, w_image):
    fused = w_text * probs_text + w_image * probs_image
    return np.argmax(fused, axis=1)


# ============================================================
# Evaluate on a given test subset
# ============================================================

def evaluate_on_subset(idx):
    y_sub = y_test[idx]
    p_text_sub = test_probs_text[idx]
    p_image_sub = test_probs_image[idx]

    # Uniform
    w_u = np.full((len(idx), 1), 0.5)
    pred_uniform = weighted_average_predict(p_text_sub, p_image_sub, w_u, w_u)
    f1_uniform = f1_score(y_sub, pred_uniform, average="weighted")

    # DPF (ITP + Gibbs, fixed beta)
    w_t_d, w_i_d = gibbs_weights(p_text_sub, p_image_sub, BETA_ITP)
    pred_dpf = weighted_average_predict(p_text_sub, p_image_sub, w_t_d, w_i_d)
    f1_dpf = f1_score(y_sub, pred_dpf, average="weighted")

    return f1_dpf, f1_uniform


# ============================================================
# Stratified bootstrap over the test set
# ============================================================

def stratified_bootstrap_indices(y, factor, seed):
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        n_sample = int(len(cls_idx) * factor)
        idx.extend(rng.choice(cls_idx, size=n_sample, replace=True))
    idx = np.asarray(idx)
    rng.shuffle(idx)
    return idx


# ============================================================
# 30 paired runs
# ============================================================

print("=" * 80)
print(f"{N_RUNS} PAIRED RUNS — WEIGHTED-AVERAGE DOWNSTREAM")
print("=" * 80)

dpf_scores = []
uniform_scores = []

for run in range(N_RUNS):
    seed = BASE_SEED + run
    idx = stratified_bootstrap_indices(y_test, BOOTSTRAP_FACTOR, seed)

    f1_dpf, f1_uniform = evaluate_on_subset(idx)

    dpf_scores.append(f1_dpf)
    uniform_scores.append(f1_uniform)

    print(
        f"Run {run + 1:02d}/{N_RUNS} | "
        f"DPF={f1_dpf:.4f} | Uniform={f1_uniform:.4f} | "
        f"Δ={f1_dpf - f1_uniform:+.4f}"
    )

dpf_scores = np.asarray(dpf_scores)
uniform_scores = np.asarray(uniform_scores)


# ============================================================
# Summary + Wilcoxon
# ============================================================

print("\n" + "=" * 80)
print("SUMMARY — WEIGHTED-AVERAGE DOWNSTREAM")
print("=" * 80)

mean_dpf = dpf_scores.mean()
sd_dpf = dpf_scores.std(ddof=1)
mean_uniform = uniform_scores.mean()
sd_uniform = uniform_scores.std(ddof=1)

diff = dpf_scores - uniform_scores
mean_diff = diff.mean()
sd_diff = diff.std(ddof=1)
se_diff = sd_diff / np.sqrt(len(diff))
tcrit = t.ppf(0.975, df=len(diff) - 1)
ci_low = mean_diff - tcrit * se_diff
ci_high = mean_diff + tcrit * se_diff

try:
    w_stat, w_p = wilcoxon(dpf_scores, uniform_scores)
except ValueError:
    w_stat, w_p = np.nan, np.nan

print(f"DPF     : {mean_dpf:.4f} ± {sd_dpf:.4f}")
print(f"Uniform : {mean_uniform:.4f} ± {sd_uniform:.4f}")
print(f"Δ (DPF - Uniform): {mean_diff:+.4f}")
print(f"95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"Wilcoxon p: {w_p:.6g}")

print("\n" + "=" * 80)
print("COMPACT REBUTTAL OUTPUT")
print("=" * 80)
print(
    f"Under the non-learned weighted-average downstream, DPF weighting "
    f"achieved {mean_dpf:.4f} vs. {mean_uniform:.4f} for uniform averaging "
    f"(Δ={mean_diff:+.4f}, 95% CI [{ci_low:+.4f}, {ci_high:+.4f}], "
    f"Wilcoxon p={w_p:.3g})."
)

R3 W4: CLASSIFIER-FREE MECHANISM PROBE
Test samples:       487
Fixed beta (ITP):   1.5
Runs:               30
Bootstrap factor:   2.0

30 PAIRED RUNS — WEIGHTED-AVERAGE DOWNSTREAM
Run 01/30 | DPF=0.4771 | Uniform=0.4839 | Δ=-0.0068
Run 02/30 | DPF=0.4641 | Uniform=0.4697 | Δ=-0.0056
Run 03/30 | DPF=0.4565 | Uniform=0.4592 | Δ=-0.0027
Run 04/30 | DPF=0.4735 | Uniform=0.4809 | Δ=-0.0074
Run 05/30 | DPF=0.4714 | Uniform=0.4787 | Δ=-0.0073
Run 06/30 | DPF=0.4589 | Uniform=0.4647 | Δ=-0.0058
Run 07/30 | DPF=0.4640 | Uniform=0.4684 | Δ=-0.0044
Run 08/30 | DPF=0.4677 | Uniform=0.4790 | Δ=-0.0114
Run 09/30 | DPF=0.4709 | Uniform=0.4884 | Δ=-0.0174
Run 10/30 | DPF=0.4777 | Uniform=0.4906 | Δ=-0.0129
Run 11/30 | DPF=0.4879 | Uniform=0.4942 | Δ=-0.0063
Run 12/30 | DPF=0.4596 | Uniform=0.4600 | Δ=-0.0004
Run 13/30 | DPF=0.4903 | Uniform=0.5006 | Δ=-0.0104
Run 14/30 | DPF=0.4662 | Uniform=0.4758 | Δ=-0.0096
Run 15/30 | DPF=0.4769 | Uniform=0.4881 | Δ=-0.0112
Run 16/30 | DPF=0.4726 | Uniform=0.4773 

# MOSI

In [14]:
"""
R3 W4: Classifier-free mechanism probe for DPF weights.
Run on both MVSA-Single and MVSA-Multiple, report side by side.

Purpose
-------
Test whether DPF's adaptive weights retain predictive utility without
a learned downstream classifier.

For both conditions:
    fused = w_text * p_text + w_image * p_image
    pred  = argmax(fused)

Uniform: w_text = w_image = 0.5
DPF:     weights from Practical ITP + Gibbs mapping

Protocol
--------
- Same precomputed upstream probabilities per dataset.
- Practical-ITP beta fixed at the validation-selected value (beta=1.5);
  no test-set tuning.
- 30 paired stratified bootstrap evaluations per dataset.
- Identical bootstrap indices for DPF and uniform within each replicate.
- Report mean ± std weighted-F1, paired difference, 95% CI,
  and Wilcoxon signed-rank test, side by side.

This is a classifier-free isolation probe, not an alternative DPF
configuration. The original DPF uses concatenation + learned downstream.
"""

import pickle
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from scipy.stats import wilcoxon, t


# ============================================================
# Configuration
# ============================================================

DATASETS = {
    "MVSA-Single": "../cache/dpf_mvsa_single_results.pkl",
    "MVSA-Multiple": "../cache/dpf_mvsa_multiple_results.pkl",
}

TOPK = 2
DELTA = 1e-2

# Fixed to the validation-selected Practical-ITP beta from the
# 30-run Table 5 experiment; no beta tuning is performed on test data.
BETA_ITP = 1.5

N_RUNS = 30
BASE_SEED = 42
BOOTSTRAP_FACTOR = 2.0


# ============================================================
# ITP
# ============================================================

def compute_itp(probs, topk=TOPK, delta=DELTA):
    probs = np.asarray(probs)
    k = min(topk, probs.shape[1])
    idx = np.argpartition(probs, -k, axis=1)[:, -k:]
    topk_probs = np.take_along_axis(probs, idx, axis=1)
    sum_topk_sq = np.sum(topk_probs ** 2, axis=1)
    max_topk_sq = np.max(topk_probs ** 2, axis=1)
    return -np.log(sum_topk_sq + delta * max_topk_sq + 1e-12)


# ============================================================
# Gibbs weights
# ============================================================

def gibbs_weights(probs_text, probs_image, beta, topk=TOPK, delta=DELTA):
    if beta == 0:
        n = len(probs_text)
        return np.full((n, 1), 0.5), np.full((n, 1), 0.5)

    e_text = compute_itp(probs_text, topk, delta)
    e_image = compute_itp(probs_image, topk, delta)

    energies = np.column_stack([e_text, e_image])
    logits = -beta * energies
    logits -= np.max(logits, axis=1, keepdims=True)

    weights = np.exp(logits)
    weights /= np.sum(weights, axis=1, keepdims=True)

    return weights[:, 0:1], weights[:, 1:2]


# ============================================================
# Weighted average + argmax (non-learned downstream)
# ============================================================

def weighted_average_predict(probs_text, probs_image, w_text, w_image):
    fused = w_text * probs_text + w_image * probs_image
    return np.argmax(fused, axis=1)


# ============================================================
# Stratified bootstrap over the test set
# ============================================================

def stratified_bootstrap_indices(y, factor, seed):
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        n_sample = int(len(cls_idx) * factor)
        idx.extend(rng.choice(cls_idx, size=n_sample, replace=True))
    idx = np.asarray(idx)
    rng.shuffle(idx)
    return idx


# ============================================================
# Run probe on one dataset
# ============================================================

def run_probe(dataset_name, pkl_path):
    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    with open(pkl_path, "rb") as f:
        results = pickle.load(f)

    test_probs_text  = np.asarray(results["test_probs_text"])
    test_probs_image = np.asarray(results["test_probs_image"])
    y_test           = np.asarray(results["y_test_int"])

    print(f"Test samples:       {len(y_test)}")
    print(f"Fixed beta (ITP):   {BETA_ITP}")
    print(f"Runs:               {N_RUNS}")
    print(f"Bootstrap factor:   {BOOTSTRAP_FACTOR}")
    print()

    dpf_scores = []
    uniform_scores = []

    for run in range(N_RUNS):
        seed = BASE_SEED + run
        idx = stratified_bootstrap_indices(y_test, BOOTSTRAP_FACTOR, seed)

        y_sub = y_test[idx]
        p_text_sub = test_probs_text[idx]
        p_image_sub = test_probs_image[idx]

        # Uniform
        w_u = np.full((len(idx), 1), 0.5)
        pred_uniform = weighted_average_predict(p_text_sub, p_image_sub, w_u, w_u)
        f1_uniform = f1_score(y_sub, pred_uniform, average="weighted")

        # DPF
        w_t_d, w_i_d = gibbs_weights(p_text_sub, p_image_sub, BETA_ITP)
        pred_dpf = weighted_average_predict(p_text_sub, p_image_sub, w_t_d, w_i_d)
        f1_dpf = f1_score(y_sub, pred_dpf, average="weighted")

        dpf_scores.append(f1_dpf)
        uniform_scores.append(f1_uniform)

        print(
            f"Run {run + 1:02d}/{N_RUNS} | "
            f"DPF={f1_dpf:.4f} | Uniform={f1_uniform:.4f} | "
            f"Δ={f1_dpf - f1_uniform:+.4f}"
        )

    dpf_scores = np.asarray(dpf_scores)
    uniform_scores = np.asarray(uniform_scores)

    mean_dpf = dpf_scores.mean()
    sd_dpf = dpf_scores.std(ddof=1)
    mean_uniform = uniform_scores.mean()
    sd_uniform = uniform_scores.std(ddof=1)

    diff = dpf_scores - uniform_scores
    mean_diff = diff.mean()
    sd_diff = diff.std(ddof=1)
    se_diff = sd_diff / np.sqrt(len(diff))
    tcrit = t.ppf(0.975, df=len(diff) - 1)
    ci_low = mean_diff - tcrit * se_diff
    ci_high = mean_diff + tcrit * se_diff

    try:
        _, w_p = wilcoxon(dpf_scores, uniform_scores)
    except ValueError:
        w_p = np.nan

    print("\n" + "-" * 80)
    print(f"SUMMARY — {dataset_name}")
    print("-" * 80)
    print(f"DPF     : {mean_dpf:.4f} ± {sd_dpf:.4f}")
    print(f"Uniform : {mean_uniform:.4f} ± {sd_uniform:.4f}")
    print(f"Δ (DPF - Uniform): {mean_diff:+.4f}")
    print(f"95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]")
    print(f"Wilcoxon p: {w_p:.6g}")

    return {
        "Dataset": dataset_name,
        "DPF_mean": mean_dpf,
        "DPF_sd": sd_dpf,
        "Uniform_mean": mean_uniform,
        "Uniform_sd": sd_uniform,
        "Delta": mean_diff,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "Wilcoxon_p": w_p,
    }


# ============================================================
# Main: run both datasets
# ============================================================

rows = []

for dataset_name, pkl_path in DATASETS.items():
    row = run_probe(dataset_name, pkl_path)
    rows.append(row)

df = pd.DataFrame(rows)


# ============================================================
# Side-by-side summary
# ============================================================

print("\n" + "=" * 80)
print("SIDE-BY-SIDE SUMMARY")
print("=" * 80)

print(
    df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# Compact rebuttal output
# ============================================================

print("\n" + "=" * 80)
print("COMPACT REBUTTAL OUTPUT")
print("=" * 80)

for row in rows:
    print(
        f"{row['Dataset']:<15} | "
        f"DPF={row['DPF_mean']:.4f} ± {row['DPF_sd']:.4f} | "
        f"Uniform={row['Uniform_mean']:.4f} ± {row['Uniform_sd']:.4f} | "
        f"Δ={row['Delta']:+.4f} | "
        f"95% CI [{row['CI_low']:+.4f}, {row['CI_high']:+.4f}] | "
        f"Wilcoxon p={row['Wilcoxon_p']:.3g}"
    )


DATASET: MVSA-Single
Test samples:       487
Fixed beta (ITP):   1.5
Runs:               30
Bootstrap factor:   2.0

Run 01/30 | DPF=0.4771 | Uniform=0.4839 | Δ=-0.0068
Run 02/30 | DPF=0.4641 | Uniform=0.4697 | Δ=-0.0056
Run 03/30 | DPF=0.4565 | Uniform=0.4592 | Δ=-0.0027
Run 04/30 | DPF=0.4735 | Uniform=0.4809 | Δ=-0.0074
Run 05/30 | DPF=0.4714 | Uniform=0.4787 | Δ=-0.0073
Run 06/30 | DPF=0.4589 | Uniform=0.4647 | Δ=-0.0058
Run 07/30 | DPF=0.4640 | Uniform=0.4684 | Δ=-0.0044
Run 08/30 | DPF=0.4677 | Uniform=0.4790 | Δ=-0.0114
Run 09/30 | DPF=0.4709 | Uniform=0.4884 | Δ=-0.0174
Run 10/30 | DPF=0.4777 | Uniform=0.4906 | Δ=-0.0129
Run 11/30 | DPF=0.4879 | Uniform=0.4942 | Δ=-0.0063
Run 12/30 | DPF=0.4596 | Uniform=0.4600 | Δ=-0.0004
Run 13/30 | DPF=0.4903 | Uniform=0.5006 | Δ=-0.0104
Run 14/30 | DPF=0.4662 | Uniform=0.4758 | Δ=-0.0096
Run 15/30 | DPF=0.4769 | Uniform=0.4881 | Δ=-0.0112
Run 16/30 | DPF=0.4726 | Uniform=0.4773 | Δ=-0.0047
Run 17/30 | DPF=0.4794 | Uniform=0.4849 | Δ=-0.005

#

In [19]:
"""
R3 W4: Classifier-free mechanism probe for DPF weights.
Run on both MVSA-Single and MVSA-Multiple, report side by side.

Purpose
-------
Test whether DPF's adaptive weights retain predictive utility without
a learned downstream classifier.

For both conditions:
    fused = w_text * p_text + w_image * p_image
    pred  = argmax(fused)

Uniform: w_text = w_image = 0.5
DPF:     weights from Practical ITP + Gibbs mapping

Protocol
--------
- Same precomputed upstream probabilities per dataset.
- Practical-ITP beta fixed at the per-dataset validation-selected value;
  no test-set tuning.
- 30 paired stratified bootstrap evaluations per dataset.
- Identical bootstrap indices for DPF and uniform within each replicate.
- Report mean ± std weighted-F1, paired difference, 95% CI,
  and Wilcoxon signed-rank test, side by side.

This is a classifier-free isolation probe, not an alternative DPF
configuration. The original DPF uses concatenation + learned downstream.
"""

import pickle
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from scipy.stats import wilcoxon, t


# ============================================================
# Configuration
# ============================================================

DATASETS = {
    "MVSA-Single": "../cache/dpf_mvsa_single_results.pkl",
    "MVSA-Multiple": "../cache/dpf_mvsa_multiple_results.pkl",
}

TOPK = 2
DELTA = 1e-2

# Per-dataset validation-selected Practical-ITP beta.
# Fill in each value from the corresponding validation beta-selection run.
# Do NOT reuse the MVSA-Single value for MVSA-Multiple.
BETA_ITP = {
    "MVSA-Single": 1.5,
    "MVSA-Multiple": 1.5,   # <-- replace with MVSA-Multiple validation-selected beta
}

N_RUNS = 30
BASE_SEED = 42
BOOTSTRAP_FACTOR = 2.0


# ============================================================
# ITP
# ============================================================

def compute_itp(probs, topk=TOPK, delta=DELTA):
    probs = np.asarray(probs)
    k = min(topk, probs.shape[1])
    idx = np.argpartition(probs, -k, axis=1)[:, -k:]
    topk_probs = np.take_along_axis(probs, idx, axis=1)
    sum_topk_sq = np.sum(topk_probs ** 2, axis=1)
    max_topk_sq = np.max(topk_probs ** 2, axis=1)
    return -np.log(sum_topk_sq + delta * max_topk_sq + 1e-12)


# ============================================================
# Gibbs weights
# ============================================================

def gibbs_weights(probs_text, probs_image, beta, topk=TOPK, delta=DELTA):
    if beta == 0:
        n = len(probs_text)
        return np.full((n, 1), 0.5), np.full((n, 1), 0.5)

    e_text = compute_itp(probs_text, topk, delta)
    e_image = compute_itp(probs_image, topk, delta)

    energies = np.column_stack([e_text, e_image])
    logits = -beta * energies
    logits -= np.max(logits, axis=1, keepdims=True)

    weights = np.exp(logits)
    weights /= np.sum(weights, axis=1, keepdims=True)

    return weights[:, 0:1], weights[:, 1:2]


# ============================================================
# Weighted average + argmax (non-learned downstream)
# ============================================================

def weighted_average_predict(probs_text, probs_image, w_text, w_image):
    fused = w_text * probs_text + w_image * probs_image
    return np.argmax(fused, axis=1)


# ============================================================
# Stratified bootstrap over the test set
# ============================================================

def stratified_bootstrap_indices(y, factor, seed):
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        n_sample = int(len(cls_idx) * factor)
        idx.extend(rng.choice(cls_idx, size=n_sample, replace=True))
    idx = np.asarray(idx)
    rng.shuffle(idx)
    return idx


# ============================================================
# Run probe on one dataset
# ============================================================

def run_probe(dataset_name, pkl_path):
    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    with open(pkl_path, "rb") as f:
        results = pickle.load(f)

    test_probs_text  = np.asarray(results["test_probs_text"])
    test_probs_image = np.asarray(results["test_probs_image"])
    y_test           = np.asarray(results["y_test_int"])

    # -------- shape / sanity checks --------
    assert len(test_probs_text) == len(test_probs_image) == len(y_test), \
        "Inconsistent test set sizes across modalities and labels."
    assert test_probs_text.shape == test_probs_image.shape, \
        "Text and image probability matrices must have the same shape."
    assert np.allclose(test_probs_text.sum(axis=1), 1.0, atol=1e-5), \
        "Text probabilities do not sum to 1."
    assert np.allclose(test_probs_image.sum(axis=1), 1.0, atol=1e-5), \
        "Image probabilities do not sum to 1."

    beta = BETA_ITP[dataset_name]

    print(f"Test samples:       {len(y_test)}")
    print(f"Fixed beta (ITP):   {beta}")
    print(f"Runs:               {N_RUNS}")
    print(f"Bootstrap factor:   {BOOTSTRAP_FACTOR}")
    print()

    dpf_scores = []
    uniform_scores = []

    for run in range(N_RUNS):
        seed = BASE_SEED + run
        idx = stratified_bootstrap_indices(y_test, BOOTSTRAP_FACTOR, seed)

        y_sub = y_test[idx]
        p_text_sub = test_probs_text[idx]
        p_image_sub = test_probs_image[idx]

        # Uniform
        w_u = np.full((len(idx), 1), 0.5)
        pred_uniform = weighted_average_predict(p_text_sub, p_image_sub, w_u, w_u)
        f1_uniform = f1_score(y_sub, pred_uniform, average="weighted")

        # DPF (per-dataset beta)
        w_t_d, w_i_d = gibbs_weights(p_text_sub, p_image_sub, beta)
        pred_dpf = weighted_average_predict(p_text_sub, p_image_sub, w_t_d, w_i_d)
        f1_dpf = f1_score(y_sub, pred_dpf, average="weighted")

        dpf_scores.append(f1_dpf)
        uniform_scores.append(f1_uniform)

        print(
            f"Run {run + 1:02d}/{N_RUNS} | "
            f"DPF={f1_dpf:.4f} | Uniform={f1_uniform:.4f} | "
            f"Δ={f1_dpf - f1_uniform:+.4f}"
        )

    dpf_scores = np.asarray(dpf_scores)
    uniform_scores = np.asarray(uniform_scores)

    mean_dpf = dpf_scores.mean()
    sd_dpf = dpf_scores.std(ddof=1)
    mean_uniform = uniform_scores.mean()
    sd_uniform = uniform_scores.std(ddof=1)

    diff = dpf_scores - uniform_scores
    mean_diff = diff.mean()
    sd_diff = diff.std(ddof=1)
    se_diff = sd_diff / np.sqrt(len(diff))
    tcrit = t.ppf(0.975, df=len(diff) - 1)
    ci_low = mean_diff - tcrit * se_diff
    ci_high = mean_diff + tcrit * se_diff

    try:
        _, w_p = wilcoxon(dpf_scores, uniform_scores)
    except ValueError:
        w_p = np.nan

    print("\n" + "-" * 80)
    print(f"SUMMARY — {dataset_name}")
    print("-" * 80)
    print(f"DPF     : {mean_dpf:.4f} ± {sd_dpf:.4f}")
    print(f"Uniform : {mean_uniform:.4f} ± {sd_uniform:.4f}")
    print(f"Δ (DPF - Uniform): {mean_diff:+.4f}")
    print(f"95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]")
    print(f"Wilcoxon p: {w_p:.6g}")

    return {
        "Dataset": dataset_name,
        "Beta": beta,
        "DPF_mean": mean_dpf,
        "DPF_sd": sd_dpf,
        "Uniform_mean": mean_uniform,
        "Uniform_sd": sd_uniform,
        "Delta": mean_diff,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "Wilcoxon_p": w_p,
    }


# ============================================================
# Main: run both datasets
# ============================================================

rows = []

for dataset_name, pkl_path in DATASETS.items():
    row = run_probe(dataset_name, pkl_path)
    rows.append(row)

df = pd.DataFrame(rows)


# ============================================================
# Side-by-side summary
# ============================================================

print("\n" + "=" * 80)
print("SIDE-BY-SIDE SUMMARY")
print("=" * 80)

print(
    df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# Compact rebuttal output
# ============================================================

print("\n" + "=" * 80)
print("COMPACT REBUTTAL OUTPUT")
print("=" * 80)

for row in rows:
    print(
        f"{row['Dataset']:<15} | "
        f"beta={row['Beta']:<4} | "
        f"DPF={row['DPF_mean']:.4f} ± {row['DPF_sd']:.4f} | "
        f"Uniform={row['Uniform_mean']:.4f} ± {row['Uniform_sd']:.4f} | "
        f"Δ={row['Delta']:+.4f} | "
        f"95% CI [{row['CI_low']:+.4f}, {row['CI_high']:+.4f}] | "
        f"Wilcoxon p={row['Wilcoxon_p']:.3g}"
    )


DATASET: MVSA-Single
Test samples:       487
Fixed beta (ITP):   1.5
Runs:               30
Bootstrap factor:   2.0

Run 01/30 | DPF=0.4771 | Uniform=0.4839 | Δ=-0.0068
Run 02/30 | DPF=0.4641 | Uniform=0.4697 | Δ=-0.0056
Run 03/30 | DPF=0.4565 | Uniform=0.4592 | Δ=-0.0027
Run 04/30 | DPF=0.4735 | Uniform=0.4809 | Δ=-0.0074
Run 05/30 | DPF=0.4714 | Uniform=0.4787 | Δ=-0.0073
Run 06/30 | DPF=0.4589 | Uniform=0.4647 | Δ=-0.0058
Run 07/30 | DPF=0.4640 | Uniform=0.4684 | Δ=-0.0044
Run 08/30 | DPF=0.4677 | Uniform=0.4790 | Δ=-0.0114
Run 09/30 | DPF=0.4709 | Uniform=0.4884 | Δ=-0.0174
Run 10/30 | DPF=0.4777 | Uniform=0.4906 | Δ=-0.0129
Run 11/30 | DPF=0.4879 | Uniform=0.4942 | Δ=-0.0063
Run 12/30 | DPF=0.4596 | Uniform=0.4600 | Δ=-0.0004
Run 13/30 | DPF=0.4903 | Uniform=0.5006 | Δ=-0.0104
Run 14/30 | DPF=0.4662 | Uniform=0.4758 | Δ=-0.0096
Run 15/30 | DPF=0.4769 | Uniform=0.4881 | Δ=-0.0112
Run 16/30 | DPF=0.4726 | Uniform=0.4773 | Δ=-0.0047
Run 17/30 | DPF=0.4794 | Uniform=0.4849 | Δ=-0.005

In [20]:
"""
Select beta for MVSA-Multiple Practical ITP.
"""

import pickle
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score

PKL_PATH = "../cache/dpf_mvsa_multiple_results.pkl"

TOPK = 2
DELTA = 1e-2
BETA_CANDIDATES = [0, 0.1, 0.25, 0.5, 0.8, 1.0, 1.1, 1.2, 1.5, 2.0, 3.0, 5.0]
BASE_SEED = 42

with open(PKL_PATH, "rb") as f:
    results = pickle.load(f)

train_probs_text  = np.asarray(results["train_probs_text"])
train_probs_image = np.asarray(results["train_probs_image"])
val_probs_text    = np.asarray(results["val_probs_text"])
val_probs_image   = np.asarray(results["val_probs_image"])
y_train = np.asarray(results["y_train_int"])
y_val   = np.asarray(results["y_val_int"])


def compute_itp(probs, topk=TOPK, delta=DELTA):
    probs = np.asarray(probs)
    k = min(topk, probs.shape[1])
    idx = np.argpartition(probs, -k, axis=1)[:, -k:]
    topk_probs = np.take_along_axis(probs, idx, axis=1)
    sum_topk_sq = np.sum(topk_probs ** 2, axis=1)
    max_topk_sq = np.max(topk_probs ** 2, axis=1)
    return -np.log(sum_topk_sq + delta * max_topk_sq + 1e-12)


def gibbs_fusion(probs_text, probs_image, beta):
    if beta == 0:
        n = len(probs_text)
        w = np.full((n, 1), 0.5)
        return np.concatenate([w * probs_text, w * probs_image], axis=1)

    e_text = compute_itp(probs_text)
    e_image = compute_itp(probs_image)
    energies = np.column_stack([e_text, e_image])
    logits = -beta * energies
    logits -= np.max(logits, axis=1, keepdims=True)
    weights = np.exp(logits)
    weights /= np.sum(weights, axis=1, keepdims=True)
    return np.concatenate([
        weights[:, 0:1] * probs_text,
        weights[:, 1:2] * probs_image,
    ], axis=1)


def make_classifier(seed):
    return xgb.XGBClassifier(
        n_estimators=350,
        max_depth=5,
        random_state=seed,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
    )


print("=" * 80)
print("MVSA-MULTIPLE: BETA SELECTION FOR PRACTICAL ITP")
print("=" * 80)

best_beta = None
best_f1 = -np.inf

for beta in BETA_CANDIDATES:
    X_train = gibbs_fusion(train_probs_text, train_probs_image, beta)
    X_val   = gibbs_fusion(val_probs_text, val_probs_image, beta)

    clf = make_classifier(BASE_SEED)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_val)
    score = f1_score(y_val, pred, average="weighted")

    print(f"beta={beta:<5} Val W-F1={score:.4f}")

    if score > best_f1:
        best_f1 = score
        best_beta = beta

print(f"\nSELECTED beta* = {best_beta} (Val W-F1 = {best_f1:.4f})")

MVSA-MULTIPLE: BETA SELECTION FOR PRACTICAL ITP
beta=0     Val W-F1=0.6919
beta=0.1   Val W-F1=0.6966
beta=0.25  Val W-F1=0.6945
beta=0.5   Val W-F1=0.6945
beta=0.8   Val W-F1=0.6954
beta=1.0   Val W-F1=0.6950
beta=1.1   Val W-F1=0.6960
beta=1.2   Val W-F1=0.6965
beta=1.5   Val W-F1=0.6975
beta=2.0   Val W-F1=0.6964
beta=3.0   Val W-F1=0.6925
beta=5.0   Val W-F1=0.6952

SELECTED beta* = 1.5 (Val W-F1 = 0.6975)


In [24]:
"""
R3 W4: Classifier-free mechanism probe for DPF weights.
Run on MVSA-Single, MVSA-Multiple, and CH-SIMS, report side by side.

Purpose
-------
Test whether DPF's adaptive weights retain predictive utility without
a learned downstream classifier.

For both conditions:
    fused = w_text * p_text + w_image * p_image
    pred  = argmax(fused)

Uniform: w_text = w_image = 0.5
DPF:     weights from Practical ITP + Gibbs mapping

Protocol
--------
- Same precomputed upstream probabilities per dataset.
- Practical-ITP beta fixed at the per-dataset validation-selected value;
  no test-set tuning.
- 30 paired stratified bootstrap evaluations per dataset.
- Identical bootstrap indices for DPF and uniform within each replicate.
- Report mean ± std weighted-F1, paired difference, 95% CI,
  and Wilcoxon signed-rank test, side by side.

This is a classifier-free isolation probe, not an alternative DPF
configuration. The original DPF uses concatenation + learned downstream.
"""

import pickle
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from scipy.stats import wilcoxon, t


# ============================================================
# Configuration
# ============================================================

DATASETS = {
    "MVSA-Single":  "../cache/dpf_mvsa_single_results.pkl",
    "MVSA-Multiple": "../cache/dpf_mvsa_multiple_results.pkl",
    "CH-SIMS":      "../cache/dpf_ch_sims_results.pkl",
}

# Per-dataset validation-selected Practical-ITP beta.
# Fill in each value from the corresponding validation beta-selection run.
BETA_ITP = {
    "MVSA-Single":  1.5,
    "MVSA-Multiple": 1.5,
    "CH-SIMS":      None,   # <-- fill after running beta selection on CH-SIMS
}

TOPK = 2
DELTA = 1e-2

N_RUNS = 30
BASE_SEED = 42
BOOTSTRAP_FACTOR = 2.0


# ============================================================
# ITP
# ============================================================

def compute_itp(probs, topk=TOPK, delta=DELTA):
    probs = np.asarray(probs)
    k = min(topk, probs.shape[1])
    idx = np.argpartition(probs, -k, axis=1)[:, -k:]
    topk_probs = np.take_along_axis(probs, idx, axis=1)
    sum_topk_sq = np.sum(topk_probs ** 2, axis=1)
    max_topk_sq = np.max(topk_probs ** 2, axis=1)
    return -np.log(sum_topk_sq + delta * max_topk_sq + 1e-12)


# ============================================================
# Gibbs weights
# ============================================================

def gibbs_weights(probs_text, probs_image, beta, topk=TOPK, delta=DELTA):
    if beta == 0:
        n = len(probs_text)
        return np.full((n, 1), 0.5), np.full((n, 1), 0.5)

    e_text = compute_itp(probs_text, topk, delta)
    e_image = compute_itp(probs_image, topk, delta)

    energies = np.column_stack([e_text, e_image])
    logits = -beta * energies
    logits -= np.max(logits, axis=1, keepdims=True)

    weights = np.exp(logits)
    weights /= np.sum(weights, axis=1, keepdims=True)

    return weights[:, 0:1], weights[:, 1:2]


# ============================================================
# Weighted average + argmax (non-learned downstream)
# ============================================================

def weighted_average_predict(probs_text, probs_image, w_text, w_image):
    fused = w_text * probs_text + w_image * probs_image
    return np.argmax(fused, axis=1)


# ============================================================
# Stratified bootstrap over the test set
# ============================================================

def stratified_bootstrap_indices(y, factor, seed):
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        n_sample = int(len(cls_idx) * factor)
        idx.extend(rng.choice(cls_idx, size=n_sample, replace=True))
    idx = np.asarray(idx)
    rng.shuffle(idx)
    return idx


# ============================================================
# Run probe on one dataset
# ============================================================

def run_probe(dataset_name, pkl_path):
    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    with open(pkl_path, "rb") as f:
        results = pickle.load(f)

    test_probs_text  = np.asarray(results["test_probs_text"])
    test_probs_image = np.asarray(results["test_probs_image"])
    y_test           = np.asarray(results["y_test_int"])

    # -------- shape / sanity checks --------
    assert len(test_probs_text) == len(test_probs_image) == len(y_test), \
        f"{dataset_name}: inconsistent test set sizes."
    assert test_probs_text.shape == test_probs_image.shape, \
        f"{dataset_name}: text and image prob matrices must have same shape."
    assert np.allclose(test_probs_text.sum(axis=1), 1.0, atol=1e-5), \
        f"{dataset_name}: text probabilities do not sum to 1."
    assert np.allclose(test_probs_image.sum(axis=1), 1.0, atol=1e-5), \
        f"{dataset_name}: image probabilities do not sum to 1."

    beta = BETA_ITP[dataset_name]
    if beta is None:
        raise ValueError(
            f"BETA_ITP['{dataset_name}'] is None. "
            f"Run beta selection on {dataset_name} validation first."
        )

    print(f"Test samples:       {len(y_test)}")
    print(f"Classes:            {len(np.unique(y_test))}")
    print(f"Fixed beta (ITP):   {beta}")
    print(f"Runs:               {N_RUNS}")
    print(f"Bootstrap factor:   {BOOTSTRAP_FACTOR}")
    print()

    dpf_scores = []
    uniform_scores = []

    for run in range(N_RUNS):
        seed = BASE_SEED + run
        idx = stratified_bootstrap_indices(y_test, BOOTSTRAP_FACTOR, seed)

        y_sub = y_test[idx]
        p_text_sub = test_probs_text[idx]
        p_image_sub = test_probs_image[idx]

        # Uniform
        w_u = np.full((len(idx), 1), 0.5)
        pred_uniform = weighted_average_predict(p_text_sub, p_image_sub, w_u, w_u)
        f1_uniform = f1_score(y_sub, pred_uniform, average="weighted")

        # DPF
        w_t_d, w_i_d = gibbs_weights(p_text_sub, p_image_sub, beta)
        pred_dpf = weighted_average_predict(p_text_sub, p_image_sub, w_t_d, w_i_d)
        f1_dpf = f1_score(y_sub, pred_dpf, average="weighted")

        dpf_scores.append(f1_dpf)
        uniform_scores.append(f1_uniform)

    dpf_scores = np.asarray(dpf_scores)
    uniform_scores = np.asarray(uniform_scores)

    mean_dpf = dpf_scores.mean()
    sd_dpf = dpf_scores.std(ddof=1)
    mean_uniform = uniform_scores.mean()
    sd_uniform = uniform_scores.std(ddof=1)

    diff = dpf_scores - uniform_scores
    mean_diff = diff.mean()
    sd_diff = diff.std(ddof=1)
    se_diff = sd_diff / np.sqrt(len(diff))
    tcrit = t.ppf(0.975, df=len(diff) - 1)
    ci_low = mean_diff - tcrit * se_diff
    ci_high = mean_diff + tcrit * se_diff

    try:
        _, w_p = wilcoxon(dpf_scores, uniform_scores)
    except ValueError:
        w_p = np.nan

    print(f"DPF     : {mean_dpf:.4f} ± {sd_dpf:.4f}")
    print(f"Uniform : {mean_uniform:.4f} ± {sd_uniform:.4f}")
    print(f"Δ       : {mean_diff:+.4f}")
    print(f"95% CI  : [{ci_low:+.4f}, {ci_high:+.4f}]")
    print(f"Wilcoxon p: {w_p:.6g}")

    return {
        "Dataset": dataset_name,
        "Beta": beta,
        "DPF_mean": mean_dpf,
        "DPF_sd": sd_dpf,
        "Uniform_mean": mean_uniform,
        "Uniform_sd": sd_uniform,
        "Delta": mean_diff,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "Wilcoxon_p": w_p,
    }


# ============================================================
# Main
# ============================================================

rows = []

for dataset_name, pkl_path in DATASETS.items():
    try:
        row = run_probe(dataset_name, pkl_path)
        rows.append(row)
    except FileNotFoundError:
        print(f"\n[SKIP] {dataset_name}: file not found at {pkl_path}")
    except KeyError as e:
        print(f"\n[SKIP] {dataset_name}: missing key {e}")
    except ValueError as e:
        print(f"\n[SKIP] {dataset_name}: {e}")

df = pd.DataFrame(rows)


# ============================================================
# Side-by-side summary
# ============================================================

print("\n" + "=" * 80)
print("SIDE-BY-SIDE SUMMARY")
print("=" * 80)

if len(df) > 0:
    print(df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
else:
    print("No datasets were successfully processed.")


# ============================================================
# Compact rebuttal output
# ============================================================

print("\n" + "=" * 80)
print("COMPACT REBUTTAL OUTPUT")
print("=" * 80)

for row in rows:
    print(
        f"{row['Dataset']:<15} | "
        f"beta={row['Beta']:<4} | "
        f"DPF={row['DPF_mean']:.4f} ± {row['DPF_sd']:.4f} | "
        f"Uniform={row['Uniform_mean']:.4f} ± {row['Uniform_sd']:.4f} | "
        f"Δ={row['Delta']:+.4f} | "
        f"95% CI [{row['CI_low']:+.4f}, {row['CI_high']:+.4f}] | "
        f"Wilcoxon p={row['Wilcoxon_p']:.3g}"
    )


DATASET: MVSA-Single
Test samples:       487
Classes:            3
Fixed beta (ITP):   1.5
Runs:               30
Bootstrap factor:   2.0

DPF     : 0.4701 ± 0.0101
Uniform : 0.4774 ± 0.0109
Δ       : -0.0073
95% CI  : [-0.0087, -0.0060]
Wilcoxon p: 1.86265e-09

DATASET: MVSA-Multiple
Test samples:       1946
Classes:            3
Fixed beta (ITP):   1.5
Runs:               30
Bootstrap factor:   2.0

DPF     : 0.6572 ± 0.0037
Uniform : 0.6583 ± 0.0035
Δ       : -0.0011
95% CI  : [-0.0013, -0.0009]
Wilcoxon p: 1.86265e-09

DATASET: CH-SIMS

[SKIP] CH-SIMS: file not found at ../cache/dpf_ch_sims_results.pkl

SIDE-BY-SIDE SUMMARY
      Dataset     Beta  DPF_mean   DPF_sd  Uniform_mean  Uniform_sd     Delta    CI_low   CI_high  Wilcoxon_p
  MVSA-Single 1.500000  0.470142 0.010072      0.477448    0.010875 -0.007306 -0.008656 -0.005957    0.000000
MVSA-Multiple 1.500000  0.657178 0.003714      0.658283    0.003458 -0.001105 -0.001288 -0.000922    0.000000

COMPACT REBUTTAL OUTPUT
MVSA-Sin

In [18]:
"""
Information Energy vs. Shannon Entropy: A Pre-DPF Empirical Analysis
--------------------------------------------------------------------
This script compares two interpretations of modality reliability:
1. Information Energy (Onicescu): higher value = more concentrated = more reliable
2. Shannon Entropy: lower value = more certain = more reliable

The analysis uses validation-set predictive probabilities from MVSA-Single.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from scipy.special import entr
import pickle

# ============================================
# Load precomputed validation predictions
# ============================================
with open("../cache/dpf_mvsa_single_results.pkl", "rb") as f:
    results = pickle.load(f)

# Validation-set predictive probabilities from ExtraTrees
val_probs_text  = results["val_probs_text"]   # (n_samples, n_classes)
val_probs_image = results["val_probs_image"]  # (n_samples, n_classes)
y_val_int       = results["y_val_int"]        # (n_samples,) ground truth

print(f"Validation set size: {len(y_val_int)} samples")
print(f"Number of classes: {val_probs_text.shape[1]}")

# ============================================
# Compute two interpretation functionals
# ============================================
# Information Energy (Onicescu): higher = more concentrated
ie_text = np.sum(val_probs_text**2, axis=1)
ie_img  = np.sum(val_probs_image**2, axis=1)

# Shannon Entropy: lower = more concentrated (more certain)
ent_text = np.sum(entr(val_probs_text), axis=1)
ent_img  = np.sum(entr(val_probs_image), axis=1)

# ============================================
# Determine the dominant class per modality
# ============================================
pred_text_class = np.argmax(val_probs_text, axis=1)
pred_img_class  = np.argmax(val_probs_image, axis=1)

# Whether each modality's prediction is correct
text_correct = (pred_text_class == y_val_int)
img_correct  = (pred_img_class == y_val_int)

# ============================================
# Core: compare the reliability rankings from IE and Entropy
# ============================================
# IE: higher IE = more concentrated = more reliable
ie_chooses_text = ie_text > ie_img

# Entropy: lower Entropy = more certain = more reliable
ent_chooses_text = ent_text < ent_img

# Agreement rate between the two interpretations
agreement = (ie_chooses_text == ent_chooses_text).mean()
disagreement = 1 - agreement

print("="*60)
print("Information Energy vs. Shannon Entropy: P(x) Analysis")
print("="*60)
print(f"Total validation samples: {len(y_val_int)}")
print(f"Classes: {val_probs_text.shape[1]}")
print()
print("--- Reliability Ranking Comparison ---")
print(f"IE and Entropy agree on which modality is more reliable: {agreement:.2%}")
print(f"Disagreement rate: {disagreement:.2%}")
print()

# ============================================
# Which interpretation aligns more closely with ground truth?
# ============================================
# For each sample: is the modality ranked as more reliable actually correct?
ie_reliable_is_correct = np.where(
    ie_chooses_text,
    text_correct,  # IE chooses text → check if text is correct
    img_correct    # IE chooses image → check if image is correct
)

ent_reliable_is_correct = np.where(
    ent_chooses_text,
    text_correct,  # Entropy chooses text → check if text is correct
    img_correct    # Entropy chooses image → check if image is correct
)

ie_reliable_rate = ie_reliable_is_correct.mean()
ent_reliable_rate = ent_reliable_is_correct.mean()

print("--- Ground Truth Comparison ---")
print(f"IE: modality ranked as more reliable is actually correct in {ie_reliable_rate:.2%}")
print(f"Entropy: modality ranked as more reliable is actually correct in {ent_reliable_rate:.2%}")
print(f"Difference (IE - Entropy): {ie_reliable_rate - ent_reliable_rate:.2%}")
print()

# ============================================
# Spearman correlation between the two functionals
# ============================================
corr_ie_ent_text, _ = spearmanr(ie_text, ent_text)
corr_ie_ent_img, _ = spearmanr(ie_img, ent_img)

print("--- Correlation between IE and Entropy ---")
print(f"Spearman correlation (text modality): {corr_ie_ent_text:.3f}")
print(f"Spearman correlation (image modality): {corr_ie_ent_img:.3f}")
print()

# ============================================
# Summary
# ============================================
print("="*60)
print("SUMMARY")
print("="*60)
if disagreement > 0.10:
    print(f"✓ IE and Entropy disagree in {disagreement:.2%} of samples → interpretations do matter")
else:
    print(f"✗ IE and Entropy almost always agree ({agreement:.2%}) → no practical difference")
    
if abs(ie_reliable_rate - ent_reliable_rate) > 0.03:
    print(f"✓ IE outperforms Entropy by {ie_reliable_rate - ent_reliable_rate:.2%}")
else:
    print(f"✗ IE and Entropy perform similarly ({ie_reliable_rate:.2%} vs {ent_reliable_rate:.2%})")

Validation set size: 487 samples
Number of classes: 3
Information Energy vs. Shannon Entropy: P(x) Analysis
Total validation samples: 487
Classes: 3

--- Reliability Ranking Comparison ---
IE and Entropy agree on which modality is more reliable: 99.18%
Disagreement rate: 0.82%

--- Ground Truth Comparison ---
IE: modality ranked as more reliable is actually correct in 53.80%
Entropy: modality ranked as more reliable is actually correct in 53.80%
Difference (IE - Entropy): 0.00%

--- Correlation between IE and Entropy ---
Spearman correlation (text modality): -0.991
Spearman correlation (image modality): -0.998

SUMMARY
✗ IE and Entropy almost always agree (99.18%) → no practical difference
✗ IE and Entropy perform similarly (53.80% vs 53.80%)
